# Phase 7 · R1 — Cross-encoder reranker

Two stages. A tuned lexical system retrieves the top-50; a cross-encoder built
on your Phase 1 checkpoint reorders them. The reranker is used for the
submission only if it beats its own first stage on dev.

In [ ]:
# ───────────────────────── EDIT THESE ─────────────────────────
OUT_OVERRIDE = None          # None -> /content/drive/MyDrive/cs781_phase7_v2
CKPT_NAME    = "best_enc.pt"  # Phase 1 ENCODER. For the Phase 1 DECODER use
                             # "best.pt" (or "best_model.pt") — it is detected
                             # automatically, nothing else has to change.
BACKBONE     = "auto"        # auto | encoder | decoder. "auto" reads the
                             # checkpoint; name one to get an error instead of
                             # a silent switch if you grab the wrong file.
DEC_ATTN     = "bi"          # DECODER ONLY. "causal" keeps the pretraining
                             # mask, so only the last token has seen the whole
                             # chunk. "bi" drops it, so every token has — much
                             # better for pooling, at the cost of a mismatch
                             # with how the model was trained. Measure both.
POOL         = None          # None -> "last" when DEC_ATTN="causal", else "mean".
                             # "bilstm" runs a BiLSTM over the hidden states:
                             # keeps the causal mask intact (no shift away from
                             # pretraining) and recovers right-context with a
                             # recurrence instead. Adds ~2.1M TRAINABLE readout
                             # parameters, reported separately from the adapter.
BILSTM_HID   = 256           # per direction; only used when POOL="bilstm"
BUDGET_MIN   = None          # None = no wall-clock cap; a run ends when its
                             # epochs finish or early stopping fires. Set a
                             # number of minutes to cap it again.
RUN          = []        # unused: the run cell is at the bottom
QUICK        = False
# ──────────────────────────────────────────────────────────────
import os, sys, json, math, time, random, re, zipfile, warnings
from pathlib import Path
from dataclasses import dataclass, asdict, field, replace
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings("ignore")

from google.colab import drive as _gdrive
_gdrive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive")
if not DRIVE.exists():
    raise RuntimeError("Drive did not mount — re-run and approve the prompt.")
OUT = Path(OUT_OVERRIDE) if OUT_OVERRIDE else DRIVE / "cs781_phase7_v2"
SEARCH = [DRIVE, Path("/content"), Path(".")]
for sub in ("cache", "models", "analysis", "plots", "submissions", "experiments"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)


def _hf_token():
    try:
        from google.colab import userdata
        t = userdata.get("HF_TOKEN")
        if t:
            print("HF token  : Colab secret"); return t
    except Exception as e:
        print(f"HF token  : secret unavailable ({type(e).__name__})")
    return os.environ.get("HF_TOKEN", "")


def find_file(name, roots=None, required=True):
    for root in (roots or SEARCH):
        root = Path(root)
        if not root.exists():
            continue
        if (root / name).exists():
            return root / name
        for p in root.rglob(name):
            return p
    if required:
        raise FileNotFoundError(f"could not find {name!r}")
    return None


TOK_PY, TOK_JSON = find_file("a1_tokenizer.py"), find_file("bpe_40k.json")
CKPT_P1 = find_file(CKPT_NAME)
sys.path.insert(0, str(TOK_PY.parent))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HF_TOKEN = _hf_token()

EV = {}
_evp = OUT / "evidence.json"
if _evp.exists():
    EV = json.loads(_evp.read_text())
    print(f"evidence  : {_evp}")
else:
    print("evidence.json not found — falling back to the settings notebook A "
          "measured as best on this corpus (16x16 chunks, max). Run A to "
          "confirm them on your own encoder.")
RET = EV.get("retrieval", {}) or {}
KQ   = int(RET.get("kq", 16)) or 16
KC   = int(RET.get("kc", 16)) or 16
AGG  = RET.get("agg", "max")
WIN  = EV.get("window", "head")
MAXL = int(RET.get("max_len", 512))
print(f"retrieval : kq={KQ} kc={KC} agg={AGG} window={WIN} max_len={MAXL}")
if EV:
    print(f"            A measured {EV['dev_micro_f1']:.4f} with this, against "
          f"{EV['single_window']:.4f} for a single window")
print(f"device    : {DEVICE}"
      + (f"  {torch.cuda.get_device_name(0)}"
         f"  {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB"
         if DEVICE.type == "cuda" else ""))
print(f"budget    : " + (f"{BUDGET_MIN} min cap" if BUDGET_MIN else
      "no wall-clock cap — epochs and early stopping decide"))
print(f"results   : one file per run in {OUT/'results.d'}, so these notebooks "
      f"can run IN PARALLEL without clobbering each other's entries")

Mounted at /content/drive
HF token  : Colab secret
evidence.json not found — falling back to the settings notebook A measured as best on this corpus (16x16 chunks, max). Run A to confirm them on your own encoder.
retrieval : kq=16 kc=16 agg=max window=head max_len=512
device    : cuda  NVIDIA A100-SXM4-40GB  42 GB
budget    : no wall-clock cap — epochs and early stopping decide
results   : one file per run in /content/drive/MyDrive/cs781_phase7_v2/results.d, so these notebooks can run IN PARALLEL without clobbering each other's entries


## Library

Identical to the other Phase 7 notebooks.

In [ ]:
@dataclass
class P7Config:
    # paths
    out_dir: str = "."
    cache_dir: str = "./cache"
    results_json: str = "./results.json"
    ckpt_p1: str = "./best_enc.pt"
    tokenizer_json: str = "./bpe_40k.json"

    # Phase 1 encoder architecture — read from the checkpoint, not guessed
    n_heads: int = 12
    dropout: float = 0.1
    rope_theta: float = 10000.0

    # Document representation. HELD CONSTANT across the experiments so a score
    # difference is attributable to the adaptation technique and not the window;
    # the window itself is ablated separately in the studies notebook.
    max_len: int = 512               # Phase 1 pretrained at 512 — this is the
                                     # RoPE table length, not a soft preference
    window: str = "head_tail"        # head | tail | head_tail
    head_frac: float = 0.5

    # --- the retrieval model ---
    pool: str = "mean"               # mean | cls | max | attn | last | bilstm
    bilstm_hidden: int = 256         # per direction; pool="bilstm" only
    backbone: str = "auto"           # auto | encoder | decoder (auto detects)
    dec_causal: bool = True          # decoder only: keep the causal mask.
                                     # False reads each chunk bidirectionally —
                                     # better for pooling, but a departure from
                                     # how the decoder was pretrained.
    sim: str = "cos"                 # cos | dot
    temperature: float = 0.05        # InfoNCE temperature on cosine similarity
    proj_dim: int = 0                # 0 = no projection head, embed = d_model

    # --- negatives ---
    negatives: str = "hard"          # inbatch | hard | mixed
    n_hard: int = 2                  # hard negatives mined per (query, positive)
    # InfoNCE only learns as much as its hardest negative. With batch_size 8 and
    # 2 hard negatives a query is asked to pick the right case out of 24, from a
    # pool of over a thousand — a task so easy the gradient says little. The
    # queue keeps recent document embeddings (detached, so no extra backward)
    # and reuses them as negatives, raising that to thousands at almost no cost.
    queue_size: int = 4096           # 0 disables the cross-batch queue
    hard_pool: int = 50              # mine from the top-N non-relevant candidates
    remine_every: int = 1            # epochs between re-mining (0 = mine once)

    # --- memory ---
    # A contrastive step encodes batch_size x (2 + n_hard) sequences at once, so
    # it is far heavier than a classification step at the same batch_size.
    # autofit probes a real step on the real GPU and shrinks these until one
    # fits; grad_checkpoint is the fallback when they cannot shrink further
    # without gutting the in-batch negatives.
    autofit: bool = True
    autofit_grow: bool = True        # also GROW the batch when the GPU allows.
                                     # batch_size sets how many in-batch
                                     # negatives each query sees, so unused VRAM
                                     # is unused training signal.
    max_batch: int = 128
    grad_checkpoint: bool = False

    # --- training ---
    lr: float = 5e-5                 # raised with the batch: more negatives per
                                     # step means a less noisy gradient, which
                                     # tolerates (and wants) a larger step
    head_lr: float = 1e-3
    weight_decay: float = 0.01
    epochs: int = 10                 # no wall-clock cap any more, so epochs
                                     # and `patience` decide when a run ends
    # Sized for an A100 and shrunk automatically on smaller cards. autofit
    # probes a real step: on 40 GB it keeps (or grows) this, on a 16 GB T4 it
    # drops to whatever fits and raises grad_accum to hold the EFFECTIVE batch
    # constant — so the learning rate stays valid either way. What a small card
    # loses is in-batch negatives, which is exactly what the queue replaces.
    batch_size: int = 32             # (query, positive) pairs per step
    grad_accum: int = 1
    warmup_frac: float = 0.06
    max_grad_norm: float = 1.0
    label_smoothing: float = 0.0
    resume: bool = True
    seed: int = 42
    eval_every: int = 100            # optimizer steps between dev evaluations
    patience: int = 3                # evals without improvement before stopping.
                                     # The BEST checkpoint is always kept, so a
                                     # tight patience costs compute, not quality
                                     # — but note the LR schedule is cosine, so
                                     # scores often improve late as it decays;
                                     # raise this if runs stop while climbing.
    max_steps: int | None = None
    max_minutes: float | None = None  # wall-clock cap; the real guard

    # --- retrieval / scoring ---
    top_k: int = 7                   # the submission format: exactly 7 ids
    embed_batch: int = 16

    # --- data size (QUICK=True for a fast smoke run) ---
    quick: bool = False
    max_train: int | None = None
    max_dev: int | None = None
    max_cand: int | None = None      # cap the candidate pool (QUICK only)

    hf_repo: str = "Exploration-Lab/CS-781-Capstone"
    hf_token: str | None = None

    def __post_init__(self):
        if self.quick:
            self.max_len = min(self.max_len, 128)
            self.max_train = self.max_train or 40
            self.max_dev = self.max_dev or 16
            self.max_cand = self.max_cand or 60
            self.epochs = 1
            self.eval_every = 10
            self.patience = 3
            self.max_steps = self.max_steps or 20
            self.hard_pool = min(self.hard_pool, 20)


def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


PAD_NAMES = ["<pad>", "[PAD]", "<PAD>", "[pad]"]


def load_tokenizer(cfg):
    """Returns (tokenizer, pad_id, vocab_size). Same resolution as Phases 2-6."""
    d = str(Path(cfg.tokenizer_json).parent)
    if d not in sys.path:
        sys.path.insert(0, d)
    from a1_tokenizer import BPETokenizer
    tok = BPETokenizer.load(str(cfg.tokenizer_json))
    base = int(tok.vocab_size)
    pad = None
    st = getattr(tok, "special_tokens", None)
    if isinstance(st, dict):
        for n in PAD_NAMES:
            if isinstance(st.get(n), int):
                pad = int(st[n]); break
    if pad is None:
        for attr in ("token_to_id", "vocab", "encoder", "stoi"):
            v = getattr(tok, attr, None)
            if isinstance(v, dict):
                for n in PAD_NAMES:
                    if isinstance(v.get(n), int):
                        pad = int(v[n]); break
            if pad is not None:
                break
    if pad is None:
        raise ValueError("Could not resolve the pad id from this tokenizer.")
    print(f"tokenizer: vocab {base:,}  pad_id {pad}")
    return tok, pad, base


def enc_ids(tok, text):
    try:
        return tok.encode(text, add_special_tokens=True)
    except TypeError:
        return tok.encode(text)

In [ ]:
def _hf(cfg, rel, tries=3):
    """Download one file, retrying transient connection failures.

    httpx raises LocalProtocolError when a pooled connection is left in a bad
    state — nothing to do with the file existing. Retrying on a fresh client
    usually succeeds, and reporting it as "file not found" sends you looking
    for the wrong problem entirely."""
    from huggingface_hub import hf_hub_download
    last = None
    for i in range(tries):
        try:
            return hf_hub_download(cfg.hf_repo, rel, repo_type="dataset",
                                   token=cfg.hf_token)
        except Exception as e:
            last = e
            name = type(e).__name__
            # a genuine 404 will not heal on a retry; a broken connection will
            if name in ("EntryNotFoundError", "RepositoryNotFoundError"):
                raise
            if i < tries - 1:
                print(f"    {rel}: {name} — retrying ({i+2}/{tries})")
                time.sleep(2 * (i + 1))
    raise last


P7_CANDIDATES = [
    "Phase_7_PCR/{sub}/{fname}",
    "Phase_7_Prior_Case_Retrieval/{sub}/{fname}",
    "Phase_7_PriorCaseRetrieval/{sub}/{fname}",
    "Phase_7/{sub}/{fname}",
]
_SUB = {"train": "Train", "dev": "Dev", "test": "Test"}


def _resolve(cfg, split, fname, local=None):
    if local and os.path.exists(local):
        return local
    errs = []
    for pat in P7_CANDIDATES:
        rel = pat.format(sub=_SUB[split], fname=fname)
        try:
            p = _hf(cfg, rel)
            print(f"  [{split}] {rel}")
            return p
        except Exception as e:
            errs.append(f"{rel}: {type(e).__name__}")
    raise FileNotFoundError(
        f"could not locate {fname!r} for the {split} split. Tried:\n  "
        + "\n  ".join(errs)
        + "\nPass local=... with the downloaded path, or add the correct pattern "
          "to P7_CANDIDATES.")


def _text_of(r):
    """The corpus stores `text` as a list of sentences; a few rows are a bare
    string. Both become one string."""
    if isinstance(r, str):
        return r
    t = r.get("text") or r.get("document") or r.get("content") or ""
    if isinstance(t, list):
        t = " ".join(map(str, t))
    return t


def _raw_cache(cfg, split, kind):
    Path(cfg.cache_dir).mkdir(parents=True, exist_ok=True)
    return Path(cfg.cache_dir) / f"p7_raw_{split}_{kind}.json.gz"


def _read_raw_cache(p):
    if not p.exists():
        return None
    import gzip
    try:
        with gzip.open(p, "rt") as f:
            return json.load(f)
    except Exception:
        return None            # a truncated cache is not a reason to fail


def _write_raw_cache(p, obj):
    import gzip
    tmp = p.with_suffix(p.suffix + ".tmp")
    with gzip.open(tmp, "wt") as f:
        json.dump(obj, f)
    tmp.replace(p)             # atomic, so an interrupted write leaves no stub


def load_queries(cfg, split, local=None):
    """Returns [(qid, text, [relevant_cand_ids] | None)], id order preserved.

    Cached to Drive on first load. Tokenized data is already cached by
    prepare_split, but the chunking notebooks need the raw text to re-window a
    document, and without this cache every one of them would re-download the
    split — and fail outright whenever the connection hiccups."""
    cache = _raw_cache(cfg, split, "queries")
    hit = _read_raw_cache(cache)
    if hit is not None:
        print(f"[cache hit ] {split} queries: {len(hit)} ({cache.name})")
        return [(q, t, r) for q, t, r in hit]
    path = _resolve(cfg, split, f"{split}.json", local)
    with open(path) as f:
        raw = json.load(f)
    out = []
    items = raw.items() if isinstance(raw, dict) else enumerate(raw)
    for qid, r in items:
        txt = _text_of(r)
        if not txt:
            continue
        rel = None
        if isinstance(r, dict):
            rel = r.get("relevant_candidates", r.get("relevant", r.get("labels")))
            if rel is not None:
                rel = [str(x) for x in rel]
        out.append((str(qid), txt, rel))
    _write_raw_cache(cache, out)
    return out


def load_candidates(cfg, split, local=None):
    """Returns [(cid, text)] — the retrieval pool for that split. Cached to
    Drive on first load, as above."""
    cache = _raw_cache(cfg, split, "candidates")
    hit = _read_raw_cache(cache)
    if hit is not None:
        print(f"[cache hit ] {split} candidates: {len(hit)} ({cache.name})")
        return [(c, t) for c, t in hit]
    path = _resolve(cfg, split, f"{split}_candidates.json", local)
    with open(path) as f:
        raw = json.load(f)
    out = []
    items = raw.items() if isinstance(raw, dict) else enumerate(raw)
    for cid, r in items:
        txt = _text_of(r)
        if txt:
            out.append((str(cid), txt))
    _write_raw_cache(cache, out)
    return out


# ---- windowing: how a long judgment becomes max_len tokens ----

def encode_window(tok, text, pad_id, cfg):
    """Returns (ids[max_len], n_real).

    Pre-truncates by words so the pure-Python BPE never processes material that
    would be discarded: encode() splits on whitespace and encodes each word
    independently, so the ids for the first N words are a prefix of the full
    document's ids (and likewise for a suffix)."""
    L, w = cfg.max_len, text.split()
    budget = 2 * L                      # every word costs >= 1 token
    if cfg.window == "head":
        ids = enc_ids(tok, " ".join(w[:budget]))[:L]
    elif cfg.window == "tail":
        ids = enc_ids(tok, " ".join(w[-budget:]))[-L:]
    elif cfg.window == "head_tail":
        n_head = int(round(L * cfg.head_frac))
        h = enc_ids(tok, " ".join(w[:budget]))
        # a document that already fits must NOT be spliced, or head and tail
        # would both cover it and the model would see the middle twice
        if len(w) <= budget and len(h) <= L:
            ids = h
        else:
            t = enc_ids(tok, " ".join(w[-budget:]))
            ids = h[:n_head] + t[-(L - n_head):]
    else:
        raise ValueError(f"unknown window {cfg.window!r}")
    n = max(len(ids), 1)
    if len(ids) < L:
        ids = ids + [pad_id] * (L - len(ids))
    return ids[:L], min(n, L)


def encode_docs(tok, texts, pad_id, cfg, desc=""):
    X = np.full((len(texts), cfg.max_len), pad_id, dtype=np.int32)
    n = np.zeros(len(texts), dtype=np.int32)
    t0 = time.time()
    for i, t in enumerate(texts):
        ids, k = encode_window(tok, t, pad_id, cfg)
        X[i] = ids; n[i] = k
        if desc and i and i % 1000 == 0:
            print(f"  {desc} {i}/{len(texts)} ({time.time()-t0:.0f}s)")
    if desc:
        print(f"  {desc} {len(texts)} docs in {time.time()-t0:.0f}s")
    return X, n


def prepare_split(cfg, tok, pad_id, split, local=None):
    """Tokenizes the queries and the candidate pool once and caches both to Drive,
    keyed by the window settings, so every later experiment and session reuses them.

    Returns a dict with:
      qid   [Nq] ids     Xq, nq   tokenized queries
      cid   [Nc] ids     Xc, nc   tokenized candidate pool
      gold  list[set[str]] | None  relevant candidate ids per query
      pos   list[list[int]] | None same, as indices into cid
    """
    local = local or {}
    Path(cfg.cache_dir).mkdir(parents=True, exist_ok=True)
    key = f"{cfg.window}_{cfg.max_len}_{cfg.head_frac:.2f}"
    # The TEST split is never subsampled, whatever the size caps say. The
    # submission must carry a prediction for every test query and must rank the
    # whole pool: dropping either would be a malformed submission, and a smoke
    # run that quietly writes one is worse than no smoke run at all.
    cap_q = None if split == "test" else (
        cfg.max_train if split == "train" else cfg.max_dev)
    cap_c = None if split == "test" else cfg.max_cand
    tag = f"{split}_{key}" + (f"_q{cap_q}" if cap_q else "") \
          + (f"_c{cap_c}" if cap_c else "")
    f = Path(cfg.cache_dir) / f"p7_{tag}.npz"
    fm = Path(cfg.cache_dir) / f"p7_{tag}.json"
    if f.exists() and fm.exists():
        z = np.load(f); meta = json.loads(fm.read_text())
        print(f"[cache hit ] {split}: {len(meta['qid'])} queries, "
              f"{len(meta['cid'])} candidates ({f.name})")
        gold = [set(g) for g in meta["gold"]] if meta["gold"] is not None else None
        return {"qid": meta["qid"], "cid": meta["cid"], "Xq": z["Xq"],
                "nq": z["nq"], "Xc": z["Xc"], "nc": z["nc"], "gold": gold,
                "pos": meta["pos"]}

    queries = load_queries(cfg, split, local.get("queries"))
    cands = load_candidates(cfg, split, local.get("candidates"))
    if cap_q and len(queries) > cap_q:
        rng = np.random.default_rng(cfg.seed)
        queries = [queries[i] for i in
                   sorted(rng.choice(len(queries), cap_q, replace=False))]
    if cap_c and len(cands) > cap_c:
        # keep every candidate that some kept query needs, then fill at random —
        # dropping a gold candidate would make the metric unreachable, not just
        # harder, and a smoke run that cannot score 1.0 hides real bugs
        need = set()
        for _, _, rel in queries:
            if rel:
                need.update(rel)
        keep = [i for i, (c, _) in enumerate(cands) if c in need]
        rest = [i for i in range(len(cands)) if i not in set(keep)]
        rng = np.random.default_rng(cfg.seed)
        extra = list(rng.permutation(rest)[: max(0, cap_c - len(keep))])
        cands = [cands[i] for i in sorted(keep + [int(x) for x in extra])]

    print(f"[tokenizing] {split}: {len(queries)} queries, "
          f"{len(cands)} candidates, window={cfg.window} max_len={cfg.max_len}")
    Xq, nq = encode_docs(tok, [t for _, t, _ in queries], pad_id, cfg,
                         desc=f"{split} queries")
    Xc, nc = encode_docs(tok, [t for _, t in cands], pad_id, cfg,
                         desc=f"{split} candidates")
    qid = [q for q, _, _ in queries]
    cid = [c for c, _ in cands]
    c_index = {c: i for i, c in enumerate(cid)}
    has_gold = any(r is not None for _, _, r in queries)
    gold = pos = None
    if has_gold:
        gold = [set(r or []) for _, _, r in queries]
        # a gold id outside this split's pool cannot be retrieved; count them
        # rather than letting them silently depress recall
        missing = sum(len(g - set(c_index)) for g in gold)
        pos = [[c_index[c] for c in sorted(g) if c in c_index] for g in gold]
        print(f"  gold: {sum(len(g) for g in gold)} relevance pairs, "
              f"{np.mean([len(g) for g in gold]):.2f} per query "
              f"(median {int(np.median([len(g) for g in gold]))})")
        if missing:
            print(f"  !! {missing} gold id(s) are not in {split}_candidates.json "
                  f"and can never be retrieved — they cap recall from above")
        empty = sum(1 for p in pos if not p)
        if empty:
            print(f"  {empty} query/queries have no retrievable positive; they are "
                  f"excluded from training but still scored")

    # temp-then-rename: a parallel run must never read a half-written cache
    _t = f.with_suffix(f".tmp{os.getpid()}.npz")
    np.savez_compressed(_t, Xq=Xq, nq=nq, Xc=Xc, nc=nc); _t.replace(f)
    _t = fm.with_suffix(f".tmp{os.getpid()}.json")
    _t.write_text(json.dumps({"qid": qid, "cid": cid, "pos": pos,
                              "gold": [sorted(g) for g in gold] if gold else None}))
    _t.replace(fm)
    return {"qid": qid, "cid": cid, "Xq": Xq, "nq": nq, "Xc": Xc, "nc": nc,
            "gold": gold, "pos": pos}


def save_notebook(dest_dir, stem="notebook"):
    """Write a copy of THIS notebook beside the run it produced.

    Colab exposes the live notebook through an internal message channel, which is
    the only reliable way to get it: a notebook opened from Drive is not at a
    predictable path, and one uploaded into the session is not in Drive at all."""
    dest = Path(dest_dir); dest.mkdir(parents=True, exist_ok=True)
    try:
        from google.colab import _message
        nb = _message.blocking_request("get_ipynb", request="", timeout_sec=120)["ipynb"]
        p = dest / f"{stem}.ipynb"
        p.write_text(json.dumps(nb, indent=1))
        print(f"  notebook copied -> {p}")
        return str(p)
    except Exception:
        pass
    print(f"  (could not capture the notebook automatically — save a copy into "
          f"{dest} by hand so the code sits with its results)")
    return None

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).to(x.dtype) * self.weight


def precompute_rope(dim, max_len, theta=10000.0, device=None):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device).float() / dim))
    t = torch.arange(max_len, device=device).float()
    ang = torch.outer(t, freqs)
    return ang.cos(), ang.sin()


def apply_rope(x, cos, sin):
    T = x.size(2)
    c = cos[:T].unsqueeze(0).unsqueeze(0)
    s = sin[:T].unsqueeze(0).unsqueeze(0)
    hd = x.size(-1) // 2
    x1, x2 = x[..., :hd], x[..., hd:]
    rot = torch.cat([-x2, x1], dim=-1)
    c_e = torch.cat([c.expand(-1, -1, -1, hd), c.expand(-1, -1, -1, hd)], -1)
    s_e = torch.cat([s.expand(-1, -1, -1, hd), s.expand(-1, -1, -1, hd)], -1)
    return x * c_e + rot * s_e


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.nh, self.hd, self.dm = cfg.n_heads, cfg.head_dim, cfg.d_model
        self.q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.v = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.o = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.scale = 1.0 / math.sqrt(cfg.head_dim)
    # Phase 1 computed the [B, heads, T, T] score matrix explicitly. That is fine
    # for a classification batch, but a contrastive step encodes the query AND
    # every document in its group at once, so B is batch_size x (2 + n_hard) and
    # the saved-for-backward score matrices alone run to several GB.
    #
    # USE_SDPA routes the same computation through scaled_dot_product_attention,
    # which never materialises the T x T matrix. It is the SAME function — no
    # parameter changes, your checkpoint still loads strictly — so this is a
    # memory optimisation, not an architecture change. Set it False to get the
    # explicit path back; the sanity notebook checks the two agree.
    USE_SDPA = True

    def forward(self, x, cos, sin, padding_mask=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if self.USE_SDPA and hasattr(F, "scaled_dot_product_attention"):
            # SDPA's bool mask marks what MAY be attended to, the opposite of
            # padding_mask, hence the negation. [B, 1, 1, T] broadcasts over
            # heads and query positions.
            am = None if padding_mask is None else \
                (~padding_mask)[:, None, None, :]
            out = F.scaled_dot_product_attention(
                q, k, v, attn_mask=am,
                dropout_p=(self.attn_drop.p if self.training else 0.0),
                scale=self.scale)
        else:
            w = torch.matmul(q, k.transpose(-2, -1)) * self.scale
            if padding_mask is not None:
                w = w.masked_fill(padding_mask.unsqueeze(1).unsqueeze(2),
                                  float("-inf"))
            w = F.softmax(w, dim=-1, dtype=torch.float32).to(q.dtype)
            w = self.attn_drop(w)
            out = torch.matmul(w, v)
        out = out.transpose(1, 2).contiguous().view(B, T, self.dm)
        return self.resid_drop(self.o(out))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up   = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model); self.attn = MultiHeadSelfAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model); self.ffn  = SwiGLU(cfg)
    def forward(self, x, cos, sin, padding_mask=None):
        x = x + self.attn(self.norm1(x), cos, sin, padding_mask)
        return x + self.ffn(self.norm2(x))


class BackboneCfg:
    pass


class LegalEncoder(nn.Module):
    """Phase 1 encoder without the MLM head."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.layers = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model)
        cos, sin = precompute_rope(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.grad_checkpoint = getattr(cfg, "grad_checkpoint", False)
        self.apply(self._init)   # only matters for tensors absent from the ckpt

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, ids, padding_mask=None):
        T = ids.size(1)
        if T > self.rope_cos.size(0):
            raise ValueError(
                f"sequence length {T} exceeds the RoPE table "
                f"({self.rope_cos.size(0)}). Phase 1 precomputed RoPE at "
                f"max_seq_len only; longer context needs the buffers rebuilt and "
                f"position interpolation.")
        x = self.drop(self.tok_emb(ids))
        for layer in self.layers:
            if self.grad_checkpoint and self.training and x.requires_grad:
                # recompute each block's activations in the backward instead of
                # storing them: roughly a third of the memory for roughly 30%
                # more compute. The lever to pull when batch_size is already as
                # small as the in-batch negatives can tolerate.
                x = torch.utils.checkpoint.checkpoint(
                    layer, x, self.rope_cos, self.rope_sin, padding_mask,
                    use_reentrant=False)
            else:
                x = layer(x, self.rope_cos, self.rope_sin, padding_mask)
        return self.norm(x)


class GQASelfAttention(nn.Module):
    """Phase 1 DECODER attention: grouped-query attention.

    `k` and `v` project to `n_kv_heads * head_dim` rather than `d_model`, and
    each KV head is shared by `n_groups` query heads. Identical to the Phase 1
    decoder's GQA module, so its weights load strictly — the one liberty taken
    is routing through SDPA, which is the same function without the explicit
    [B, heads, T, T] matrix.

    `causal` is a property of this USE of the model, not of the checkpoint. A
    causal model reading a 512-token chunk lets token i see only 0..i, so a mean
    over positions is dominated by tokens that saw almost nothing. Turning it off
    makes every position see the whole chunk, which is what an embedding wants —
    at the cost of a mismatch with how the model was pretrained. Both are
    exposed because only the dev score can say which wins here."""

    USE_SDPA = True

    def __init__(self, cfg):
        super().__init__()
        self.nqh = cfg.n_heads
        self.nkh = getattr(cfg, "n_kv_heads", cfg.n_heads)
        self.ng = self.nqh // self.nkh
        self.hd, self.dm = cfg.head_dim, cfg.d_model
        self.q = nn.Linear(cfg.d_model, self.nqh * self.hd, bias=False)
        self.k = nn.Linear(cfg.d_model, self.nkh * self.hd, bias=False)
        self.v = nn.Linear(cfg.d_model, self.nkh * self.hd, bias=False)
        self.o = nn.Linear(self.nqh * self.hd, cfg.d_model, bias=False)
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.scale = 1.0 / math.sqrt(cfg.head_dim)
        self.causal = bool(getattr(cfg, "causal", True))

    def forward(self, x, cos, sin, padding_mask=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nqh, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.nkh, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.nkh, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if self.ng > 1:
            k = k.unsqueeze(2).expand(B, self.nkh, self.ng, T, self.hd
                                      ).reshape(B, self.nqh, T, self.hd)
            v = v.unsqueeze(2).expand(B, self.nkh, self.ng, T, self.hd
                                      ).reshape(B, self.nqh, T, self.hd)
        am = None
        if padding_mask is not None:
            am = (~padding_mask)[:, None, None, :]          # keys that are real
        if self.causal:
            cm = torch.ones(T, T, dtype=torch.bool, device=x.device).tril()
            am = cm[None, None] if am is None else (am & cm[None, None])
        if self.USE_SDPA and hasattr(F, "scaled_dot_product_attention"):
            out = F.scaled_dot_product_attention(
                q, k, v, attn_mask=am,
                dropout_p=(self.attn_drop.p if self.training else 0.0),
                scale=self.scale)
        else:
            w = torch.matmul(q, k.transpose(-2, -1)) * self.scale
            if am is not None:
                w = w.masked_fill(~am, float("-inf"))
            w = F.softmax(w, dim=-1, dtype=torch.float32).to(q.dtype)
            out = torch.matmul(self.attn_drop(w), v)
        out = out.transpose(1, 2).contiguous().view(B, T, self.nqh * self.hd)
        return self.resid_drop(self.o(out))


class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model); self.attn = GQASelfAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model); self.ffn = SwiGLU(cfg)

    def forward(self, x, cos, sin, padding_mask=None):
        x = x + self.attn(self.norm1(x), cos, sin, padding_mask)
        return x + self.ffn(self.norm2(x))


class LegalDecoder(LegalEncoder):
    """Phase 1 decoder without the LM head. Same embedding, RoPE, RMSNorm,
    SwiGLU and forward pass as the encoder — only the attention block differs,
    so everything downstream (pooling, PEFT, distillation) is shared."""

    def __init__(self, cfg):
        nn.Module.__init__(self)
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.layers = nn.ModuleList([DecoderBlock(cfg)
                                     for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model)
        cos, sin = precompute_rope(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.grad_checkpoint = getattr(cfg, "grad_checkpoint", False)
        self.apply(self._init)


def detect_backbone(sd):
    """'decoder' if the checkpoint has grouped KV heads or a causal LM head."""
    d = sd["tok_emb.weight"].shape[1]
    kw = sd.get("layers.0.attn.k.weight")
    gqa = kw is not None and kw.shape[0] != d
    lm = "lm_head.weight" in sd and "mlm_dense.weight" not in sd
    return "decoder" if (gqa or lm) else "encoder"


def backbone_cfg_from_sd(sd, n_heads=12, max_seq_len=512, dropout=0.1,
                         rope_theta=10000.0, grad_checkpoint=False,
                         causal=True):
    c = BackboneCfg()
    c.vocab_size, c.d_model = sd["tok_emb.weight"].shape
    c.n_layers = max(int(m.group(1)) for k in sd
                     if (m := re.match(r"layers\.(\d+)\.", k))) + 1
    c.d_ff = sd["layers.0.ffn.gate.weight"].shape[0]
    c.n_heads, c.max_seq_len, c.dropout = n_heads, max_seq_len, dropout
    c.rope_theta = rope_theta
    c.grad_checkpoint = grad_checkpoint
    c.causal = causal
    if c.d_model % c.n_heads:
        raise ValueError(f"d_model {c.d_model} not divisible by n_heads {c.n_heads}")
    c.head_dim = c.d_model // c.n_heads
    kw = sd.get("layers.0.attn.k.weight")
    c.n_kv_heads = c.n_heads if kw is None else kw.shape[0] // c.head_dim
    if c.n_heads % c.n_kv_heads:
        raise ValueError(f"n_heads {c.n_heads} not divisible by n_kv_heads "
                         f"{c.n_kv_heads}")
    return c

In [ ]:
class AttnPool(nn.Module):
    """A single learned query vector scoring each position. The only pooling
    option with parameters of its own, which is why it is a study and not the
    default: it would otherwise add trainable weights to a 'frozen backbone' run
    and muddy the parameter counts."""
    def __init__(self, d_model):
        super().__init__()
        self.w = nn.Linear(d_model, 1, bias=False)
        nn.init.normal_(self.w.weight, std=0.02)
    def forward(self, h, keep):
        s = self.w(h).squeeze(-1)
        s = s.masked_fill(keep.squeeze(-1) == 0, float("-inf"))
        a = F.softmax(s, dim=-1).unsqueeze(-1)
        return (h * a).sum(1)


class BiLSTMPool(nn.Module):
    """A bidirectional LSTM readout over the backbone's hidden states.

    Why this exists, on a CAUSAL backbone: dropping the causal mask
    (`DEC_ATTN="bi"`) buys bidirectional context but shifts every attention
    distribution away from what the model was pretrained on. This keeps the mask
    — the transformer runs exactly as trained — and recovers right-context with
    a recurrence on top instead. It is also a real fix for the causal readout
    problem: `last` stakes a 512-token document on one token that is often a
    fragment, and `mean` over causal states is dominated by positions that saw
    almost nothing.

    The cost is honest and reported separately: these are randomly initialised
    parameters trained on a few thousand pairs, and they dwarf what IA3 (1.5k)
    or LoRMA (33k) move. Being POOLING, they are held fixed across every
    technique, so the comparison between techniques stays valid — but the
    "adapts with N parameters" headline must quote the adapter count, not this.

    Padding is handled with a packed sequence, which matters more here than
    anywhere else in this file: run naively over a padded batch, the BACKWARD
    direction would start inside the padding and walk through it before
    reaching any real token, so every document's reverse state would be
    contaminated by however much padding it happened to have."""

    def __init__(self, d_model, hidden=256, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(d_model, hidden, num_layers=1, batch_first=True,
                            bidirectional=True)
        self.out_dim = 2 * hidden
        self.drop = nn.Dropout(dropout)

    def forward(self, h, keep):
        lens = keep.squeeze(-1).sum(1).long().clamp(min=1).cpu()
        # cuDNN's fused LSTM under autocast is a well-known source of silent
        # dtype trouble with packed sequences; it is tiny next to the
        # transformer, so run it in fp32 and spend the memory.
        with torch.amp.autocast(device_type=h.device.type, enabled=False):
            x = h.float()
            p = nn.utils.rnn.pack_padded_sequence(
                x, lens, batch_first=True, enforce_sorted=False)
            out, _ = self.lstm(p)
            out, _ = nn.utils.rnn.pad_packed_sequence(
                out, batch_first=True, total_length=x.size(1))
        k = keep.to(out.dtype)
        return self.drop((out * k).sum(1) / k.sum(1).clamp(min=1))


class PCREncoder(nn.Module):
    """Phase 1 encoder -> pooled document vector -> (optional) projection.

    `embed()` returns L2-normalised vectors when sim='cos', so a dot product IS
    the cosine and the whole pool can be ranked with one matrix multiply."""
    def __init__(self, backbone, d_model, cfg):
        super().__init__()
        self.backbone = backbone
        self.pool_kind = cfg.pool
        self.sim = cfg.sim
        self.drop = nn.Dropout(cfg.dropout)
        self.attn_pool = AttnPool(d_model) if cfg.pool == "attn" else None
        self.lstm_pool = (BiLSTMPool(d_model, cfg.bilstm_hidden, cfg.dropout)
                          if cfg.pool == "bilstm" else None)
        d_out = d_model if self.lstm_pool is None else self.lstm_pool.out_dim
        if cfg.proj_dim:
            self.proj = nn.Linear(d_out, cfg.proj_dim, bias=False)
            nn.init.normal_(self.proj.weight, std=0.02)
            self.dim = cfg.proj_dim
        else:
            self.proj = None
            self.dim = d_out
        if self.lstm_pool is not None:
            n = sum(p.numel() for p in self.lstm_pool.parameters())
            print(f"  BiLSTM readout: hidden {cfg.bilstm_hidden}/direction -> "
                  f"{self.lstm_pool.out_dim}d, {n:,} trainable parameters "
                  f"(POOLING — held fixed across techniques, counted separately "
                  f"from the adapter)")

    def pooled(self, ids, pad_mask):
        h = self.backbone(ids, pad_mask)
        keep = (~pad_mask).unsqueeze(-1).to(h.dtype)
        if self.pool_kind == "cls":
            return h[:, 0]
        if self.pool_kind == "max":
            return h.masked_fill(keep == 0, -1e4).max(1).values
        if self.pool_kind == "last":
            # the natural readout for a CAUSAL model: the last real token is the
            # only position that has seen the whole chunk.
            idx = keep.squeeze(-1).sum(1).long().clamp(min=1) - 1
            return h[torch.arange(h.size(0), device=h.device), idx]
        if self.pool_kind == "attn":
            return self.attn_pool(h, keep)
        if self.pool_kind == "bilstm":
            return self.lstm_pool(h, keep)
        return (h * keep).sum(1) / keep.sum(1).clamp(min=1)

    def forward(self, ids, pad_mask):
        z = self.pooled(ids, pad_mask)
        if self.proj is not None:
            z = self.proj(self.drop(z))
        return F.normalize(z, dim=-1) if self.sim == "cos" else z


def _load_ckpt_sd(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    sd = {(k[7:] if k.startswith("module.") else k): v
          for k, v in (ck["model"] if "model" in ck else ck).items()}
    return ck, sd


def load_phase1(cfg, n_layers_keep=None, verbose=True):
    """Builds the bi-encoder on the Phase 1 encoder checkpoint.
    `n_layers_keep` < depth takes an evenly spaced subset of blocks, which is how
    the distillation student is built."""
    ck, sd = _load_ckpt_sd(cfg.ckpt_p1)

    # ---- which Phase 1 model is this? ----
    found = detect_backbone(sd)
    want = getattr(cfg, "backbone", "auto")
    if want not in ("auto", "encoder", "decoder"):
        raise ValueError(f"backbone must be auto|encoder|decoder, got {want!r}")
    if want != "auto" and want != found:
        raise ValueError(
            f"BACKBONE is {want!r} but {Path(cfg.ckpt_p1).name} is the Phase 1 "
            f"{found.upper()} "
            + ("(grouped KV heads / causal LM head). "
               if found == "decoder" else "(full MHA, no LM head). ")
            + f"Set BACKBONE='{found}' or point CKPT_NAME at your "
              f"{want} checkpoint.")
    kind = found
    bc = backbone_cfg_from_sd(sd, n_heads=cfg.n_heads, max_seq_len=cfg.max_len,
                              dropout=cfg.dropout, rope_theta=cfg.rope_theta,
                              grad_checkpoint=cfg.grad_checkpoint,
                              causal=bool(getattr(cfg, "dec_causal", True)))
    keep_idx = list(range(bc.n_layers))
    if n_layers_keep is not None and n_layers_keep < bc.n_layers:
        step = bc.n_layers / n_layers_keep
        keep_idx = [min(bc.n_layers - 1, int(round(i * step)))
                    for i in range(n_layers_keep)]
        bc.n_layers = n_layers_keep

    bb = (LegalDecoder if kind == "decoder" else LegalEncoder)(bc)
    want, new = bb.state_dict(), {}
    for k in want:
        if k.startswith("layers."):
            i = int(k.split(".")[1])
            src = k.replace(f"layers.{i}.", f"layers.{keep_idx[i]}.", 1)
        else:
            src = k
        if src in sd and sd[src].shape == want[k].shape:
            new[k] = sd[src]
    missing = [k for k in want if k not in new and
               k not in ("rope_cos", "rope_sin")]
    bb.load_state_dict(new, strict=False)

    if verbose:
        if kind == "decoder":
            print(f"GQA DECODER (causal LM-pretrained, read "
                  f"{'CAUSALLY' if bc.causal else 'BIDIRECTIONALLY'}): "
                  f"{bc.d_model}d x {bc.n_layers}L, d_ff={bc.d_ff}, "
                  f"{bc.n_heads}Qh/{bc.n_kv_heads}KVh, "
                  f"vocab={bc.vocab_size:,}")
        else:
            print(f"MHA ENCODER (bidirectional, MLM-pretrained): {bc.d_model}d x "
                  f"{bc.n_layers}L, d_ff={bc.d_ff}, vocab={bc.vocab_size:,}")
        print(f"  loaded {len(new)}/{len(want)} tensors "
              f"(step {ck.get('step','?')}, val_loss {ck.get('val_loss','?')})")
        if keep_idx != list(range(len(keep_idx))):
            print(f"  layer subset: {keep_idx}")
        if missing:
            print(f"  NOT loaded (fresh init): {missing}")
        std = bb.tok_emb.weight.std().item()
        print(f"  tok_emb std = {std:.4f} (fresh init ~0.0200)")
        if abs(std - 0.02) < 0.002:
            print("  !! near init — check that the checkpoint really loaded")

    model = PCREncoder(bb, bc.d_model, cfg)
    print(f"  bi-encoder: pool={cfg.pool} sim={cfg.sim} dim={model.dim}")
    ck = {k: v for k, v in ck.items() if k not in ("model", "opt", "scaler")}
    del sd
    return model, bc, ck

In [ ]:
# ════════════════════════════════════════════ bookkeeping

def param_report(model, label=""):
    tot = sum(p.numel() for p in model.parameters())
    # a 4-bit weight is still a weight: NF4Linear holds it in a uint8 buffer, not
    # a Parameter, so without this the QLoRA row would claim a model 7x smaller
    # than it is and its "% trainable" would be meaningless
    tot += sum(m.n for m in model.modules() if m.__class__.__name__ == "NF4Linear")
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{label}trainable {tr:,} / {tot:,} = {100*tr/max(tot,1):.3f}%")
    return tr, tot


def freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False


def trainable_names(model):
    return [n for n, p in model.named_parameters() if p.requires_grad]


def _blocks(model):
    return model.backbone.layers


# ════════════════════════════════════════════ A4 Q4.3  LoRA

class LoRALinear(nn.Module):
    """h = W₀x + (α/r)·B A x.  W₀ frozen; A [r, d_in] Kaiming-uniform,
    B [d_out, r] zeros, so ΔW = 0 at step 0 and training starts at exactly the
    pretrained function."""
    def __init__(self, base: nn.Linear, r=8, alpha=16, p_drop=0.0):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.r, self.alpha = r, alpha
        self.scaling = alpha / r
        self.lora_A = nn.Parameter(torch.zeros(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))   # B stays 0
        self.drop = nn.Dropout(p_drop)

    def forward(self, x):
        return (self.base(x)
                + (self.drop(x) @ self.lora_A.T) @ self.lora_B.T * self.scaling)

    @torch.no_grad()
    def merged_copy(self):
        """An independent nn.Linear with the scaled update applied exactly once.
        Merge into a COPY, never into the shared pretrained object."""
        new = nn.Linear(self.base.in_features, self.base.out_features,
                        bias=self.base.bias is not None)
        new.weight.copy_(self.base.weight + self.scaling * (self.lora_B @ self.lora_A))
        if self.base.bias is not None:
            new.bias.copy_(self.base.bias)
        return new


# ════════════════════════════════════════════ A4 Q4.4  LoRMA (multiplicative)

LORMA_VARIANT = {
    "name": "LoRMA+",
    "equation": "W' = ((alpha/r) B A + I_dout) @ W0   (additive rank inflation)",
    "paper": "Bihany, Patel & Modi, arXiv:2506.07621v1, Eq. 8",
}


class LoRMALinear(nn.Module):
    """A4 Q4.4 — LoRMA₊, multiplicative adaptation with additive rank inflation.

    LoRA ADDS a low-rank matrix to W₀; LoRMA MULTIPLIES W₀ by one. A naive
    (BA)·W₀ has rank ≤ r and so cannot even represent "no change", because the
    identity has full rank — that is the rank-inhibition problem. LoRMA₊ restores
    full rank by adding the identity:

        W' = ((α/r)·B A + I_{d_out}) · W₀,   B ∈ R^{d_out×r},  A ∈ R^{r×d_out}

    Note the shapes: both factors are keyed to d_out, because the update
    multiplies W₀ from the LEFT. That is the detail LoRA code gets wrong when it
    is adapted to LoRMA by search-and-replace — LoRA's A is [r, d_in].

    The forward below is the **re-ordered** form: it never materialises the
    d_out × d_out product, it applies the factors to the activations instead,
        h = y + (α/r)·B(A y),   y = W₀x
    which is the same function at a fraction of the cost. `merged_copy` is the
    dense form, kept for the equality check and for merged-latency measurement.

    B = 0 at init with A random, so W' = W₀ exactly at step 0."""
    def __init__(self, base: nn.Linear, r=8, alpha=16, p_drop=0.0):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        d_out = base.out_features
        self.r, self.alpha = r, alpha
        self.scaling = alpha / r
        self.lorma_A = nn.Parameter(torch.zeros(r, d_out))
        self.lorma_B = nn.Parameter(torch.zeros(d_out, r))
        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))   # B stays 0
        self.drop = nn.Dropout(p_drop)

    def forward(self, x):                      # re-ordered form
        y = self.base(x)
        return y + (self.drop(y) @ self.lorma_A.T) @ self.lorma_B.T * self.scaling

    @torch.no_grad()
    def merged_copy(self):                     # dense form: W' = (sBA + I) W0
        d_out = self.base.out_features
        W = (self.base.weight if isinstance(self.base, nn.Linear)
             else nf4_dequantize(self.base.packed, self.base.scales,
                                 self.base.shape, self.base.n, self.base.block))
        M = self.scaling * (self.lorma_B @ self.lorma_A) + torch.eye(
            d_out, device=self.lorma_B.device, dtype=W.dtype)
        new = nn.Linear(self.base.in_features, d_out,
                        bias=getattr(self.base, "bias", None) is not None)
        new.weight.copy_(M @ W)
        if new.bias is not None:
            new.bias.copy_(M @ self.base.bias)
        return new


def lorma_rank_report(module, tol=1e-6):
    """Numerical ranks via SVD: the factors, the naive product B A (rank ≤ r),
    and the inflated transformation (α/r)BA + I (full rank). This is the evidence
    for the A4 Q4.4(a) argument, and it is worth printing once in the report."""
    with torch.no_grad():
        BA = (module.lorma_B @ module.lorma_A).double()
        M = module.scaling * BA + torch.eye(BA.size(0), dtype=torch.float64)
        W0 = (module.base.weight if isinstance(module.base, nn.Linear)
              else nf4_dequantize(module.base.packed, module.base.scales,
                                  module.base.shape, module.base.n,
                                  module.base.block)).double()
        dW = (M @ W0) - W0
        rk = lambda T: int((torch.linalg.svdvals(T) > tol).sum())
        return {"r": module.r, "rank_B": rk(module.lorma_B.double()),
                "rank_A": rk(module.lorma_A.double()), "rank_BA": rk(BA),
                "rank_inflated": rk(M), "rank_update_W": rk(dW),
                "d_out": BA.size(0)}


# ════════════════════════════════════════════ A4 Q4.2  bottleneck adapters

class BottleneckAdapter(nn.Module):
    """h' = h + W_up GELU(W_down h + b_down) + b_up.  W_up and b_up are
    zero-initialised, so the adapted model is EXACTLY the frozen model at step 0.
    `enabled=False` returns the input unchanged, which is how the layer-ablation
    study switches a trained adapter off without retraining."""
    def __init__(self, d_model, bottleneck=32):
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.down = nn.Linear(d_model, bottleneck)
        self.up = nn.Linear(bottleneck, d_model)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
        self.enabled = True
    def forward(self, h):
        if not self.enabled:
            return h
        return h + self.up(F.gelu(self.down(self.norm(h))))


class AdaptedBlock(nn.Module):
    """Frozen encoder block with adapters. `variant='houlsby'` puts one after each
    sublayer (attention and FFN); `'pfeiffer'` puts one after the block only, which
    is A4's single post-block placement and roughly half the parameters."""
    def __init__(self, block: EncoderBlock, d_model, bottleneck=32,
                 variant="pfeiffer"):
        super().__init__()
        self.block = block
        self.variant = variant
        self.adapter = BottleneckAdapter(d_model, bottleneck)
        self.adapter_attn = (BottleneckAdapter(d_model, bottleneck)
                             if variant == "houlsby" else None)
    def forward(self, x, cos, sin, padding_mask=None):
        b = self.block
        if self.adapter_attn is None:
            return self.adapter(b(x, cos, sin, padding_mask))
        x = x + self.adapter_attn(b.attn(b.norm1(x), cos, sin, padding_mask))
        return x + self.adapter(b.ffn(b.norm2(x)))


# ════════════════════════════════════════════ IA³  (Liu et al., "Few-Shot PEFT is
# Better and Cheaper than In-Context Learning")

class IA3Linear(nn.Module):
    """h = ℓ ⊙ W₀x, with ℓ a learned per-output-feature scale initialised to 1.

    The cheapest adapter on the task statement's list: it rescales existing
    features instead of adding new directions, so it costs d_out parameters per
    wrapped projection and nothing at inference once folded into W₀. Applied to
    k, v and the FFN up-projection — scaling `up`'s output rescales the SwiGLU
    intermediate, which is where IA³ acts in the paper."""
    def __init__(self, base: nn.Linear):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.ia3 = nn.Parameter(torch.ones(base.out_features))
    def forward(self, x):
        return self.base(x) * self.ia3

    @torch.no_grad()
    def merged_copy(self):
        new = nn.Linear(self.base.in_features, self.base.out_features,
                        bias=self.base.bias is not None)
        new.weight.copy_(self.base.weight * self.ia3[:, None])
        if self.base.bias is not None:
            new.bias.copy_(self.base.bias * self.ia3)
        return new


# ════════════════════════════════════════════ A4 Q5.1  NF4 quantization (QLoRA)

# The 16 NF4 levels: quantiles of a standard normal, normalised to [-1, 1], with
# an exact zero. A4 Q5.1 derives these; they are reproduced here as a constant.
NF4_LEVELS = torch.tensor([
    -1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
    -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
    0.07958029955625534, 0.16093020141124725, 0.24611230194568634,
    0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
    0.7229568362236023, 1.0], dtype=torch.float32)


def nf4_quantize(W: torch.Tensor, block=64):
    """Blockwise absmax NF4. Two 4-bit indices share one byte, so the stored
    tensor really is 4 bits per weight and the memory number in the report is
    honest."""
    flat = W.detach().float().reshape(-1)
    n = flat.numel()
    pad = (-n) % block
    if pad:
        flat = torch.cat([flat, torch.zeros(pad, device=flat.device)])
    blocks = flat.view(-1, block)
    scales = blocks.abs().amax(dim=1).clamp(min=1e-12)
    norm = blocks / scales[:, None]
    idx = (norm[..., None] - NF4_LEVELS.to(W.device)).abs().argmin(-1).to(torch.uint8)
    idx = idx.reshape(-1)
    packed = (idx[0::2] << 4) | idx[1::2]
    return packed, scales.half(), tuple(W.shape), n, block


def nf4_dequantize(packed, scales, shape, n, block, dtype=torch.float32):
    hi = (packed >> 4).to(torch.long)
    lo = (packed & 0x0F).to(torch.long)
    idx = torch.stack([hi, lo], dim=1).reshape(-1)
    vals = NF4_LEVELS.to(packed.device)[idx].view(-1, block)
    out = (vals * scales.float()[:, None]).reshape(-1)[:n]
    return out.view(shape).to(dtype)


class NF4Linear(nn.Module):
    """A frozen nn.Linear whose weight lives in 4 bits and is dequantized inside
    the forward. It holds no trainable parameters, which is the point:
    QLoRA = this plus a LoRA branch in full precision on top."""
    def __init__(self, base: nn.Linear, block=64):
        super().__init__()
        packed, scales, shape, n, blk = nf4_quantize(base.weight, block)
        self.register_buffer("packed", packed)
        self.register_buffer("scales", scales)
        self.shape, self.n, self.block = shape, n, blk
        self.in_features, self.out_features = base.in_features, base.out_features
        if base.bias is not None:
            self.register_buffer("bias", base.bias.detach().clone())
        else:
            self.bias = None
        with torch.no_grad():
            W = nf4_dequantize(packed, scales, shape, n, blk)
            self.quant_err = float((W - base.weight).abs().mean())
            self.rel_err = float((W - base.weight).norm() / base.weight.norm())

    def stored_bytes(self):
        return self.packed.numel() + self.scales.numel() * 2

    def forward(self, x):
        W = nf4_dequantize(self.packed, self.scales, self.shape, self.n,
                           self.block, dtype=x.dtype)
        return F.linear(x, W, self.bias)


# ════════════════════════════════════════════ injection helpers

ATTN_TARGETS = ("q", "k", "v", "o")
FFN_TARGETS = ("gate", "up", "down")


def _wrappable(m):
    """A module LoRA can wrap: a real nn.Linear, or an NF4Linear (which is what
    makes QLoRA QLoRA rather than 'LoRA with a quantized sibling')."""
    return ((isinstance(m, nn.Linear) or m.__class__.__name__ == "NF4Linear")
            and not isinstance(m, (LoRALinear, LoRMALinear, IA3Linear)))


def inject_lora(model, r=8, alpha=16, targets=ATTN_TARGETS, ffn_targets=(),
                p_drop=0.0):
    injected = []
    for blk in _blocks(model):
        for holder, names in ((blk.attn, targets), (blk.ffn, ffn_targets)):
            for name in names:
                base = getattr(holder, name)
                if not _wrappable(base):
                    continue
                new = LoRALinear(base, r, alpha, p_drop)
                setattr(holder, name, new); injected.append(new)
    if not injected:
        raise RuntimeError(
            "LoRA wrapped nothing. Either the target names are wrong for this "
            "architecture, or the projections were already replaced by something "
            "_wrappable() does not recognise.")
    return injected


def inject_lorma(model, r=8, alpha=16, targets=ATTN_TARGETS, ffn_targets=(),
                 p_drop=0.0):
    injected = []
    for blk in _blocks(model):
        for holder, names in ((blk.attn, targets), (blk.ffn, ffn_targets)):
            for name in names:
                base = getattr(holder, name)
                if not _wrappable(base):
                    continue
                new = LoRMALinear(base, r, alpha, p_drop)
                setattr(holder, name, new); injected.append(new)
    if not injected:
        raise RuntimeError("LoRMA wrapped nothing — check the target names.")
    return injected


def inject_ia3(model):
    """k, v and the FFN up-projection, per the IA³ paper."""
    injected = []
    for blk in _blocks(model):
        for holder, names in ((blk.attn, ("k", "v")), (blk.ffn, ("up",))):
            for name in names:
                base = getattr(holder, name)
                if not _wrappable(base):
                    continue
                new = IA3Linear(base)
                setattr(holder, name, new); injected.append(new)
    if not injected:
        raise RuntimeError("IA3 wrapped nothing — check the target names.")
    return injected


def inject_adapters(model, bottleneck=32, variant="pfeiffer"):
    bb = model.backbone
    d = bb.cfg.d_model
    bb.layers = nn.ModuleList(
        [AdaptedBlock(b, d, bottleneck, variant) for b in bb.layers])
    out = []
    for b in bb.layers:
        out.append(b.adapter)
        if b.adapter_attn is not None:
            out.append(b.adapter_attn)
    return out


def quantize_nf4(model, targets=ATTN_TARGETS, ffn_targets=FFN_TARGETS, block=64):
    """Replaces the frozen projections with NF4 storage. Call BEFORE inject_lora
    so the LoRA branch wraps the quantized layer."""
    q, fp_bytes, q_bytes, errs = [], 0, 0, []
    for blk in _blocks(model):
        for holder, names in ((blk.attn, targets), (blk.ffn, ffn_targets)):
            for name in names:
                base = getattr(holder, name)
                if not isinstance(base, nn.Linear):
                    continue
                fp_bytes += base.weight.numel() * 4
                new = NF4Linear(base, block)
                q_bytes += new.stored_bytes()
                errs.append(new.rel_err)
                setattr(holder, name, new); q.append(new)
    print(f"NF4: {len(q)} projections  {fp_bytes/1e6:.1f} MB fp32 -> "
          f"{q_bytes/1e6:.1f} MB packed  ({fp_bytes/max(q_bytes,1):.2f}x smaller), "
          f"mean relative weight error {np.mean(errs):.4f}")
    return q, {"fp32_mb": fp_bytes / 1e6, "nf4_mb": q_bytes / 1e6,
               "ratio": fp_bytes / max(q_bytes, 1), "rel_err": float(np.mean(errs))}


def set_partial_trainable(model, last_k=4, train_norm=True,
                          train_embeddings=False):
    """Layer-wise fine-tuning: unfreeze the top `last_k` blocks (plus the final
    norm), leaving the lower blocks and the embedding table frozen."""
    freeze_all(model)
    bb = model.backbone
    n = len(bb.layers)
    for i in range(max(0, n - last_k), n):
        for p in bb.layers[i].parameters():
            p.requires_grad = True
    if train_norm:
        for p in bb.norm.parameters():
            p.requires_grad = True
    if train_embeddings:
        bb.tok_emb.weight.requires_grad = True
    if model.proj is not None:
        for p in model.proj.parameters():
            p.requires_grad = True
    if model.attn_pool is not None:
        for p in model.attn_pool.parameters():
            p.requires_grad = True
    return list(range(max(0, n - last_k), n))


def unfreeze_readout(model):
    """Whatever sits above the backbone — the projection and the attention pool
    if they exist. A PEFT run trains these too, because freezing a randomly
    initialised projection would cripple every technique equally and tell you
    nothing about the technique."""
    for m in (model.proj, model.attn_pool, model.lstm_pool):
        if m is not None:
            for p in m.parameters():
                p.requires_grad = True


def readout_params(model):
    """Trainable parameters that belong to the READOUT, not to the adaptation
    technique. Reported separately so 'IA3 adapts with 1,536 parameters' stays
    a true statement when a BiLSTM pool is in use."""
    n = 0
    for m in (getattr(model, "proj", None), getattr(model, "attn_pool", None),
              getattr(model, "lstm_pool", None)):
        if m is not None:
            n += sum(p.numel() for p in m.parameters() if p.requires_grad)
    return n

In [ ]:
def embed_all(model, X, n, pad_id, cfg, desc="", batch=None):
    """[N, dim] float32 embeddings for a tokenized document array."""
    model.eval()
    bs = batch or cfg.embed_batch
    out, t0 = [], time.time()
    with torch.no_grad():
        for i in range(0, len(X), bs):
            ids = torch.from_numpy(X[i:i + bs].astype(np.int64)).to(DEVICE)
            pm = ids == pad_id
            with torch.amp.autocast(device_type=DEVICE.type,
                                    enabled=(DEVICE.type == "cuda")):
                z = model(ids, pm)
            out.append(z.float().cpu())
            if desc and i and (i // bs) % 100 == 0:
                print(f"  {desc} {i}/{len(X)} ({time.time()-t0:.0f}s)")
    Z = torch.cat(out).numpy()
    if desc:
        print(f"  {desc}: {len(X)} docs -> {Z.shape} in {time.time()-t0:.0f}s")
    return Z


def rank_all(Zq, Zc, k=7, block=512):
    """Top-k candidate indices per query, highest similarity first.
    Blocked over queries so a large pool never materialises an [Nq, Nc] matrix."""
    Q = torch.from_numpy(Zq); C = torch.from_numpy(Zc)
    k = min(k, C.shape[0])
    out = []
    for i in range(0, len(Q), block):
        s = Q[i:i + block] @ C.T
        out.append(s.topk(k, dim=1).indices)
    return torch.cat(out).numpy()


def micro_f1_at_k(topk, gold_sets, cid, k=7, pos=None):
    """The leaderboard metric.

    `gold_sets` are the FULL gold id sets, including any id that is not in this
    split's candidate pool — that is how the evaluator computes recall, so a gold
    id the pool does not contain counts against you and is not quietly dropped.
    `pos` (retrievable indices only) gives the attainable ceiling alongside."""
    tp = 0
    for i, g in enumerate(gold_sets):
        tp += sum(1 for j in topk[i][:k] if cid[j] in g)
    n_pred = k * len(gold_sets)
    n_gold = sum(len(g) for g in gold_sets)
    P = tp / max(n_pred, 1)
    R = tp / max(n_gold, 1)
    F1 = 2 * P * R / max(P + R, 1e-12)
    out = {"micro_f1": F1, "micro_precision": P, "micro_recall": R,
           "tp": int(tp), "n_pred": int(n_pred), "n_gold": int(n_gold)}
    if pos is not None:
        # the best any ranker could do: min(k, |retrievable gold|) per query
        best = sum(min(k, len(p)) for p in pos)
        bP = best / max(n_pred, 1); bR = best / max(n_gold, 1)
        out["ceiling_micro_f1"] = 2 * bP * bR / max(bP + bR, 1e-12)
        out["ceiling_recall"] = bR
    return out


def recall_at(topk, gold_sets, cid, ks=(1, 5, 7, 10, 20)):
    """Recall@k for the error analysis: how much of the gold the ranker finds at
    depths you are not allowed to submit, which separates 'cannot rank' from
    'cannot rank inside 7'."""
    out = {}
    for k in ks:
        if k > topk.shape[1]:
            continue
        tp = sum(sum(1 for j in topk[i][:k] if cid[j] in g)
                 for i, g in enumerate(gold_sets))
        out[f"recall@{k}"] = tp / max(sum(len(g) for g in gold_sets), 1)
    return out


def evaluate(model, data, pad_id, cfg, max_queries=None, desc=""):
    """Embeds the pool and the queries, ranks, scores. Returns (metrics, topk)."""
    Zc = embed_all(model, data["Xc"], data["nc"], pad_id, cfg, desc=desc and "pool")
    Xq, gold, pos = data["Xq"], data["gold"], data["pos"]
    if max_queries and len(Xq) > max_queries:
        Xq, gold, pos = Xq[:max_queries], gold[:max_queries], pos[:max_queries]
    Zq = embed_all(model, Xq, data["nq"], pad_id, cfg)
    topk = rank_all(Zq, Zc, k=max(cfg.top_k, 20))
    m = micro_f1_at_k(topk, gold, data["cid"], cfg.top_k, pos)
    m.update(recall_at(topk, gold, data["cid"]))
    return m, topk


# ════════════════════════════════════════════ hard negatives

def mine_hard_negatives(model, data, pad_id, cfg, Zc=None):
    """For each (query, positive) pair, the highest-scoring candidates that are
    NOT relevant to that query. Returns a list parallel to the pair list."""
    Zc = Zc if Zc is not None else embed_all(model, data["Xc"], data["nc"],
                                             pad_id, cfg, desc="mining pool")
    Zq = embed_all(model, data["Xq"], data["nq"], pad_id, cfg)
    depth = min(cfg.hard_pool + max(len(p) for p in data["pos"]) + 1, len(Zc))
    topk = rank_all(Zq, Zc, k=depth)
    negs = []
    for qi, plist in enumerate(data["pos"]):
        bad = set(plist)
        cand = [int(j) for j in topk[qi] if j not in bad]
        negs.append(cand[: cfg.hard_pool])
    n_found = float(np.mean([len(x) for x in negs])) if negs else 0.0
    print(f"  mined hard negatives: {n_found:.0f} per query from the top "
          f"{depth} of a {len(Zc)}-candidate pool")
    return negs, Zc


def build_pairs(data):
    """One training example per (query, relevant candidate). A query with three
    relevant priors becomes three examples, which is the standard construction and
    weights a query by how much supervision it carries."""
    pairs = []
    for qi, plist in enumerate(data["pos"] or []):
        for pj in plist:
            pairs.append((qi, pj))
    return pairs


class PairDataset(torch.utils.data.Dataset):
    """Yields (query_ids, positive_ids, [negative_ids...]). Negatives are drawn
    fresh each epoch from the mined pool, so the model does not see the same
    three hard cases every time."""
    def __init__(self, data, pairs, negs, cfg, seed=42):
        self.data, self.pairs, self.negs, self.cfg = data, pairs, negs, cfg
        self.rng = np.random.default_rng(seed)
        self.n_neg = cfg.n_hard if cfg.negatives in ("hard", "mixed") else 0
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        qi, pj = self.pairs[i]
        q = torch.from_numpy(self.data["Xq"][qi].astype(np.int64))
        d = [torch.from_numpy(self.data["Xc"][pj].astype(np.int64))]
        didx = [pj]
        if self.n_neg:
            pool = self.negs[qi] if self.negs else []
            if len(pool) >= self.n_neg:
                pick = self.rng.choice(len(pool), self.n_neg, replace=False)
                chosen = [pool[int(x)] for x in pick]
            else:
                # not enough mined negatives (a tiny pool, or a query whose gold
                # covers most of it): fall back to uniform random non-positives
                bad = set(self.data["pos"][qi])
                chosen = []
                while len(chosen) < self.n_neg:
                    r = int(self.rng.integers(len(self.data["Xc"])))
                    if r not in bad:
                        chosen.append(r)
            for c in chosen:
                d.append(torch.from_numpy(self.data["Xc"][c].astype(np.int64)))
                didx.append(int(c))
        # the indices come along so distillation can look up precomputed teacher
        # embeddings instead of running the teacher inside the training loop
        return q, torch.stack(d), qi, torch.tensor(didx, dtype=torch.long)


def make_pair_loader(data, pairs, negs, cfg, shuffle=True, seed=42):
    g = torch.Generator(); g.manual_seed(seed)
    return torch.utils.data.DataLoader(
        PairDataset(data, pairs, negs, cfg, seed), batch_size=cfg.batch_size,
        shuffle=shuffle, num_workers=2, pin_memory=(DEVICE.type == "cuda"),
        generator=g, drop_last=True)


class NegativeQueue:
    """FIFO of recent document embeddings, reused as extra InfoNCE negatives.

    The embeddings are detached, so this costs one matmul and no extra backward
    pass — a batch of 8 can be scored against thousands of negatives instead of
    24. This is the MoCo / cross-batch-memory trick, and it is the standard
    answer to "my contrastive batch is too small to fit on the GPU".

    Each entry carries its document index so an entry that is a POSITIVE for the
    current query can be masked out. Without that masking the loss would push a
    query away from a case it is supposed to retrieve — a false negative, and
    with 8 gold cases per query on this corpus they would not be rare."""

    def __init__(self, dim, size, device):
        self.Z = torch.zeros(size, dim, device=device)
        self.idx = torch.full((size,), -1, dtype=torch.long, device=device)
        self.size, self.n, self.ptr = size, 0, 0

    @torch.no_grad()
    def add(self, Z, idx):
        Z, idx = Z.detach().float(), idx.to(self.idx.device)
        B = Z.size(0)
        if B >= self.size:                      # a batch larger than the queue
            Z, idx, B = Z[-self.size:], idx[-self.size:], self.size
        end = self.ptr + B
        if end <= self.size:
            self.Z[self.ptr:end] = Z; self.idx[self.ptr:end] = idx
        else:                                   # wrap around
            k = self.size - self.ptr
            self.Z[self.ptr:] = Z[:k]; self.idx[self.ptr:] = idx[:k]
            self.Z[:end - self.size] = Z[k:]; self.idx[:end - self.size] = idx[k:]
        self.ptr = end % self.size
        self.n = min(self.n + B, self.size)

    def view(self):
        return self.Z[:self.n], self.idx[:self.n]


def contrastive_loss(Zq, Zd, cfg, n_per_q, queue=None, pos_lists=None,
                     doc_idx=None):
    """InfoNCE over in-batch + hard negatives.

    Zq  [B, dim]                 the queries
    Zd  [B * n_per_q, dim]       doc d*n_per_q is query d's positive; the rest of
                                 its slots are its hard negatives
    Every document in the batch is a negative for every other query, which is
    what makes in-batch negatives nearly free: the similarity matrix is computed
    once and reused B times."""
    s = (Zq @ Zd.T) / cfg.temperature
    labels = torch.arange(Zq.size(0), device=Zq.device) * n_per_q

    # ---- in-batch false negatives ----
    # InfoNCE treats every other document in the batch as a negative, but with
    # ~6 gold cases per query on this corpus, some of them ARE relevant to this
    # query. At batch 32 that is ~4 cells per step and 93% of batches, and it
    # grows as B^2 — so a larger batch buys more negatives AND more wrong ones.
    # Mask those cells, never the query's own positive.
    if pos_lists is not None and doc_idx is not None:
        bad = torch.zeros_like(s, dtype=torch.bool)
        di = doc_idx.reshape(-1).to(s.device)
        for i, p in enumerate(pos_lists):
            if len(p):
                bad[i] = torch.isin(di, p)
        bad[torch.arange(Zq.size(0), device=s.device), labels] = False
        s = s.masked_fill(bad, float("-inf"))

    if queue is not None and queue.n > 0:
        Zqu, iqu = queue.view()
        sq = (Zq @ Zqu.T.to(Zq.dtype)) / cfg.temperature
        if pos_lists is not None:
            # mask any queued document that is relevant to THIS query
            bad = torch.zeros_like(sq, dtype=torch.bool)
            for i, p in enumerate(pos_lists):
                if len(p):
                    bad[i] = torch.isin(iqu, p)
            sq = sq.masked_fill(bad, float("-inf"))
        s = torch.cat([s, sq], dim=1)
    return F.cross_entropy(s, labels, label_smoothing=cfg.label_smoothing), s

In [ ]:
def _param_groups(model, cfg):
    """The readout (projection / attention pool) learns faster than the backbone:
    it is randomly initialised, the backbone is not."""
    readout, backbone = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (readout if n.startswith(("proj", "attn_pool")) else backbone).append(p)
    groups = []
    if backbone:
        groups.append({"params": backbone, "lr": cfg.lr,
                       "weight_decay": cfg.weight_decay})
    if readout:
        groups.append({"params": readout, "lr": cfg.head_lr,
                       "weight_decay": cfg.weight_decay})
    return groups


def _probe_step(model, bs, npq, cfg, pad_id):
    """One real forward+backward at this shape. Raises OutOfMemoryError if it
    does not fit, which is the whole point."""
    V = model.backbone.tok_emb.num_embeddings
    q = torch.randint(4, V, (bs, cfg.max_len), device=DEVICE)
    d = torch.randint(4, V, (bs * npq, cfg.max_len), device=DEVICE)
    model.train()
    with torch.amp.autocast(device_type=DEVICE.type, enabled=True):
        loss, _ = contrastive_loss(model(q, q == pad_id),
                                   model(d, d == pad_id), cfg, npq)
    loss.backward()
    model.zero_grad(set_to_none=True)
    del q, d, loss
    torch.cuda.synchronize()


def autofit_batch(model, cfg, pad_id):
    """Shrink the step until it fits, keeping the effective batch constant.

    A contrastive step is batch_size x (2 + n_hard) sequences through the
    encoder with gradients, so it is several times heavier than a
    classification step at the same batch_size — which is why a config that
    worked in Phase 5 can OOM here. This probes the real GPU before training
    starts, rather than letting a run die forty minutes in.

    Hard negatives are given up before batch size is, because batch size also
    sets how many IN-BATCH negatives each query sees, and those are the cheap
    ones: halving batch_size costs negatives twice over."""
    if DEVICE.type != "cuda" or not cfg.autofit:
        return cfg
    eff = cfg.batch_size * cfg.grad_accum
    use_hard = cfg.negatives in ("hard", "mixed")

    # GROWING comes first. For contrastive training, batch_size is not just a
    # throughput knob: it sets how many in-batch negatives each query is scored
    # against, and gradient accumulation cannot substitute for it — each
    # micro-batch gets only its own small negative set. So spend spare VRAM on a
    # bigger batch, trading grad_accum down to keep the effective batch steady.
    trials = []
    if cfg.autofit_grow:
        g_bs, g_ga = cfg.batch_size, cfg.grad_accum
        while g_ga > 1 and g_bs * 2 <= cfg.max_batch:
            g_bs, g_ga = g_bs * 2, max(1, g_ga // 2)
            trials.append((g_bs, cfg.n_hard))
        while g_bs * 2 <= cfg.max_batch:       # grad_accum already 1
            g_bs *= 2
            trials.append((g_bs, cfg.n_hard))
        trials.reverse()                        # largest first

    bs, nh = cfg.batch_size, cfg.n_hard
    while True:
        trials.append((bs, nh))
        if use_hard and nh > 1:
            nh -= 1
        elif bs > 1:
            bs //= 2
        else:
            break
    for ckpt in (cfg.grad_checkpoint, True):
        model.backbone.grad_checkpoint = ckpt
        for bs, nh in trials:
            npq = 1 + (nh if use_hard else 0)
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            try:
                _probe_step(model, bs, npq, cfg, pad_id)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                continue
            except RuntimeError as e:
                if "out of memory" not in str(e).lower():
                    raise
                torch.cuda.empty_cache()
                continue
            pk = torch.cuda.max_memory_allocated() / 1e9
            tt = torch.cuda.get_device_properties(0).total_memory / 1e9
            torch.cuda.empty_cache()
            ga = max(1, round(eff / bs)) if bs <= cfg.batch_size else \
                 max(1, cfg.batch_size * cfg.grad_accum // bs)
            if (bs, nh, ckpt) != (cfg.batch_size, cfg.n_hard, cfg.grad_checkpoint):
                print(f"  autofit: batch_size {cfg.batch_size} -> {bs}, "
                      f"n_hard {cfg.n_hard} -> {nh}"
                      + (", gradient checkpointing ON" if ckpt else "")
                      + f", grad_accum -> {ga} (effective batch {bs*ga} kept)")
            print(f"  step fits: {bs} queries + {bs*npq} documents = "
                  f"{bs*(1+npq)} sequences, peak {pk:.1f}/{tt:.0f} GB")
            return replace(cfg, batch_size=bs, n_hard=nh, grad_accum=ga,
                           grad_checkpoint=ckpt)
        if not ckpt:
            print("  autofit: nothing fit — retrying with gradient checkpointing")
    raise RuntimeError(
        "even one query with one document and gradient checkpointing will not "
        f"fit on this GPU at max_len={cfg.max_len}. Lower cfg.max_len (attention "
        f"is quadratic in it) — 256 uses about a quarter of the memory.")


def find_prior(cfg, name):
    """A checkpoint for this experiment already on Drive, from an earlier session."""
    cands = [Path(cfg.out_dir) / "models" / f"{name}.pt"]
    try:
        cands += sorted(DRIVE.glob(f"**/cs781_phase7/models/{name}.pt"))
    except Exception:
        pass
    for p in cands:
        if p.exists():
            return p
    return None


def planned_steps(cfg, n_pairs):
    spe = max(1, n_pairs // (cfg.batch_size * cfg.grad_accum))
    total = spe * cfg.epochs
    return min(total, cfg.max_steps) if cfg.max_steps else total


def resume_into(model, cfg, name, n_pairs):
    """Load a prior checkpoint for `name` and return (start_step, skip).

    Call AFTER any PEFT injection, so the saved adapter tensors have somewhere to
    land. `skip=True` means the prior run already finished its schedule."""
    if not cfg.resume:
        return 0, False
    p = find_prior(cfg, name)
    if p is None:
        return 0, False
    try:
        ck = torch.load(p, map_location="cpu", weights_only=False)
    except Exception as e:
        print(f"  (prior checkpoint unreadable: {type(e).__name__}) — starting fresh")
        return 0, False
    sd = ck.get("model", ck)
    cur = model.state_dict()
    use = {k: v for k, v in sd.items()
           if k in cur and cur[k].shape == v.shape}
    model.load_state_dict(use, strict=False)
    step = int(ck.get("step", 0))
    total = planned_steps(cfg, n_pairs)
    best = ck.get("metrics", {}).get("micro_f1")
    print(f"  resuming {name} from {p.name}: {len(use)}/{len(cur)} tensors, "
          f"step {step}/{total}" + (f", dev micro-F1 {best:.4f}" if best else ""))
    if step >= total:
        print(f"  that run already covered its schedule — rerun with "
              f"cfg.epochs raised, or resume=False, to train it further")
        return step, True
    return step, False


def train(model, data_tr, data_dev, pairs, negs, pad_id, cfg, label="run",
          start_step=0, kd=None):
    """Contrastive training with in-batch + hard negatives.

    `kd`, when given, is a dict {"Zq": [Nq, d], "Zc": [Nc, d], "alpha": float,
    "tau": float} of PRECOMPUTED teacher embeddings. The teacher's score for any
    pair in the batch is then a dot product of two cached vectors, so distillation
    costs one pass over the corpus up front instead of a second forward pass on
    every step.

    Returns (best_metrics, history). The model is left holding its BEST weights,
    not its last."""
    model.to(DEVICE)
    total = planned_steps(cfg, len(pairs))
    opt = torch.optim.AdamW(_param_groups(model, cfg))
    warm = max(1, int(cfg.warmup_frac * total))

    def lr_at(s):
        if s < warm:
            return s / warm
        p = (s - warm) / max(1, total - warm)
        return 0.5 * (1 + math.cos(math.pi * min(p, 1.0)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)
    for _ in range(start_step):
        sched.step()
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))
    n_per_q = 1 + (cfg.n_hard if cfg.negatives in ("hard", "mixed") else 0)

    # positives per training query, as GPU tensors, so the queue's false
    # negatives can be masked without rebuilding them every step
    POS = [torch.as_tensor(p, dtype=torch.long, device=DEVICE)
           for p in (data_tr["pos"] or [])]
    queue = (NegativeQueue(model.dim, cfg.queue_size, DEVICE)
             if cfg.queue_size else None)

    best, best_sd, hist = None, None, []
    step, bad, t0, stop = start_step, 0, time.time(), False
    loader = make_pair_loader(data_tr, pairs, negs, cfg, seed=cfg.seed)
    _neg = cfg.batch_size * n_per_q
    print(f"  {label}: {len(pairs)} pairs, {total} steps "
          f"(batch {cfg.batch_size} x accum {cfg.grad_accum}, "
          f"{n_per_q} docs/query), eval every {cfg.eval_every}"
          + (f", cap {cfg.max_minutes:.0f} min" if cfg.max_minutes else ""))
    print(f"  negatives per query: {_neg-1} in-batch"
          + (f" + up to {cfg.queue_size} queued = {_neg-1+cfg.queue_size:,}"
             if queue is not None else "")
          + f"  (pool is {len(data_tr['cid']):,} candidates)")

    for ep in range(cfg.epochs):
        if stop:
            break
        if ep and cfg.remine_every and ep % cfg.remine_every == 0 \
                and cfg.negatives in ("hard", "mixed"):
            print(f"  [epoch {ep}] re-mining hard negatives under the current model")
            negs, _ = mine_hard_negatives(model, data_tr, pad_id, cfg)
            loader = make_pair_loader(data_tr, pairs, negs, cfg,
                                      seed=cfg.seed + ep)
            model.train()
        model.train()
        opt.zero_grad(set_to_none=True)
        for bi, (q, d, qidx, didx) in enumerate(loader):
            q = q.to(DEVICE, non_blocking=True)
            d = d.to(DEVICE, non_blocking=True)
            B, npq, L = d.shape
            d = d.reshape(B * npq, L)
            with torch.amp.autocast(device_type=DEVICE.type,
                                    enabled=(DEVICE.type == "cuda")):
                Zq = model(q, q == pad_id)
                Zd = model(d, d == pad_id)
                loss, s = contrastive_loss(
                    Zq, Zd, cfg, npq, queue=queue, doc_idx=didx,
                    pos_lists=([POS[int(q)] for q in qidx] if POS else None))
                if kd is not None:
                    # the teacher's view of the SAME similarity row. Matching
                    # the full row — not just the positive — is what transfers:
                    # it tells the student which wrong candidates the teacher
                    # thought were nearly right, which is the ranking knowledge.
                    #
                    # The row now includes the queued negatives, and the teacher
                    # must cover exactly the same columns: its embeddings are
                    # precomputed for the whole corpus, so extending it is free,
                    # and KD over thousands of documents is a far richer signal
                    # than KD over the 24 in the batch.
                    cols = didx.reshape(-1).to(DEVICE)
                    if queue is not None and queue.n > 0:
                        cols = torch.cat([cols, queue.view()[1]])
                    tq = kd["Zq"][qidx].to(DEVICE)
                    tc = kd["Zc"][cols.cpu()].to(DEVICE)
                    with torch.no_grad():
                        st = (tq @ tc.T) / kd["tau"]
                        # the student masked its false negatives to -inf; the
                        # teacher must mask the same columns, or the two
                        # distributions are defined over different supports
                        st = st.masked_fill(torch.isinf(s) & (s < 0),
                                            float("-inf"))
                    # Put BOTH sides on the same temperature. `s` already
                    # carries 1/cfg.temperature from the InfoNCE term, so
                    # dividing it by tau again would make the student ~20x
                    # sharper than the teacher and the KL would measure that
                    # scale mismatch instead of any ranking disagreement.
                    s_kd = s * cfg.temperature / kd["tau"]
                    kl = F.kl_div(F.log_softmax(s_kd, dim=-1),
                                  F.log_softmax(st, dim=-1),
                                  reduction="batchmean", log_target=True)
                    loss = (1 - kd["alpha"]) * loss + kd["alpha"] * kl
            scaler.scale(loss / cfg.grad_accum).backward()
            if queue is not None:
                # enqueue AFTER the backward: these are this step's documents,
                # detached, to serve as negatives for the steps that follow
                queue.add(Zd, didx.reshape(-1))
            if (bi + 1) % cfg.grad_accum:
                continue
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                cfg.max_grad_norm)
            scaler.step(opt); scaler.update()
            opt.zero_grad(set_to_none=True); sched.step()
            step += 1

            if step % cfg.eval_every == 0 or step >= total:
                m, _ = evaluate(model, data_dev, pad_id, cfg)
                model.train()
                hist.append({"step": step, "loss": float(loss.item()), **m})
                mark = ""
                if best is None or m["micro_f1"] > best["micro_f1"]:
                    best, bad = m, 0
                    best_sd = {k: v.detach().cpu().clone()
                               for k, v in model.state_dict().items()}
                    mark = "  <- best"
                else:
                    bad += 1
                print(f"    step {step:>5}/{total}  loss {loss.item():.4f}  "
                      f"dev micro-F1@{cfg.top_k} {m['micro_f1']:.4f}  "
                      f"R@7 {m.get('recall@7', 0):.3f}  "
                      f"({(time.time()-t0)/60:.1f} min){mark}")
                if bad >= cfg.patience:
                    print(f"    early stop: {bad} evals without improvement")
                    stop = True; break
            if step >= total:
                stop = True; break
            if cfg.max_minutes and (time.time() - t0) / 60 >= cfg.max_minutes:
                print(f"    wall-clock cap reached ({cfg.max_minutes:.0f} min) "
                      f"at step {step} — stopping on the best checkpoint so far")
                stop = True; break

    if best is None:                       # capped before the first evaluation
        best, _ = evaluate(model, data_dev, pad_id, cfg)
        best_sd = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        hist.append({"step": step, **best})
    if best_sd is not None:
        model.load_state_dict(best_sd)
    best["steps_trained"] = step
    best["minutes"] = (time.time() - t0) / 60
    return best, hist

In [ ]:
def _results_dir(cfg):
    d = Path(cfg.out_dir) / "results.d"
    d.mkdir(parents=True, exist_ok=True)
    return d


def load_results(cfg):
    """Merge the per-run files with any legacy single results.json.

    Each run writes its OWN file, so notebooks can run in parallel without the
    load-modify-write race that a shared results.json has: two processes
    finishing at once would each read the file before the other wrote, and one
    entry would vanish. Per-run files cannot collide — a run only ever writes
    its own name."""
    out = {}
    p = Path(cfg.results_json)
    if p.exists():
        try:
            out.update(json.loads(p.read_text()))
        except Exception:
            pass
    for f in sorted(_results_dir(cfg).glob("*.json")):
        try:
            out[f.stem] = json.loads(f.read_text())
        except Exception:
            pass                      # a half-written file from a live run
    return out


def already_done(cfg, name):
    r = load_results(cfg).get(name)
    if not r:
        return False
    ck = Path(cfg.out_dir) / "models" / f"{name}.pt"
    if not ck.exists():
        print(f"[{name}] in results.json but the checkpoint is gone — rerunning")
        return False
    print(f"[{name}] already done: dev micro-F1@7 "
          f"{r['metrics']['micro_f1']:.4f}  (skipping)")
    return True


def state_bytes(sd):
    return sum(v.numel() * v.element_size() for v in sd.values()
               if hasattr(v, "numel"))


def trainable_state(model):
    keep = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v for k, v in model.state_dict().items() if k in keep}


def save_run(cfg, name, model, bc, technique, info, metrics, tr, tot,
             extra=None):
    d = Path(cfg.out_dir) / "models"; d.mkdir(parents=True, exist_ok=True)
    torch.save({"model": model.state_dict(), "backbone_cfg": vars(bc),
                "technique": technique, "info": info, "metrics": metrics,
                "step": metrics.get("steps_trained", 0),
                "cfg": asdict(cfg), "trainable": tr, "total": tot,
                **(extra or {})}, d / f"{name}.pt")
    return d / f"{name}.pt"


def measure_latency(model, pad_id, cfg, batch=8, reps=8, warmup=2):
    """Milliseconds to embed one document — the inference-cost column of the
    efficiency table, which the Expected Analysis asks for explicitly."""
    model.eval()
    # sample ids from the model's OWN vocabulary — a hardcoded ceiling indexes
    # out of the embedding table on any model whose vocab is smaller
    V = model.backbone.tok_emb.num_embeddings
    x = torch.randint(4, V, (batch, cfg.max_len), device=DEVICE)
    pm = x == pad_id
    with torch.no_grad():
        for _ in range(warmup):
            model(x, pm)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(reps):
            model(x, pm)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
    return (time.time() - t0) / (reps * batch) * 1000


def _atomic_write(path, text):
    """Write via a temp file and rename, so a reader never sees a partial file
    and two writers cannot interleave. Matters once runs go in parallel."""
    path = Path(path)
    tmp = path.with_suffix(path.suffix + f".tmp{os.getpid()}")
    tmp.write_text(text)
    tmp.replace(path)


def record(cfg, name, technique, info, metrics, tr, tot, costs=None, extra=None):
    row = {"technique": technique, "info": info, "metrics": metrics,
           "trainable": int(tr), "total": int(tot),
           "pct_trainable": 100 * tr / max(tot, 1),
           "costs": costs or {}, **(extra or {})}
    # this run's own file — no other process writes it
    _atomic_write(_results_dir(cfg) / f"{name}.json",
                  json.dumps(row, indent=2, default=float))
    # and a merged copy for convenience; harmless if a parallel run races it,
    # because the per-run files above are the source of truth
    res = load_results(cfg)
    res[name] = row
    try:
        _atomic_write(cfg.results_json, json.dumps(res, indent=2, default=float))
    except Exception:
        pass
    print(f"[{name}] recorded: dev micro-F1@7 {metrics['micro_f1']:.4f}  "
          f"(P {metrics['micro_precision']:.4f} / R {metrics['micro_recall']:.4f}), "
          f"{tr:,} trainable ({100*tr/max(tot,1):.3f}%)")
    return res


def save_dev_analysis(cfg, name, data, topk, metrics):
    """The dev ranking, so the analysis notebook can study failures without
    re-running any model."""
    d = Path(cfg.out_dir) / "analysis"; d.mkdir(parents=True, exist_ok=True)
    rows = []
    for i, qid in enumerate(data["qid"][:len(topk)]):
        got = [data["cid"][j] for j in topk[i][:cfg.top_k]]
        gold = sorted(data["gold"][i]) if data["gold"] else []
        rows.append({"qid": qid, "top7": got, "gold": gold,
                     "hits": len(set(got) & set(gold))})
    (d / f"{name}_dev.json").write_text(json.dumps(
        {"metrics": metrics, "rows": rows}, indent=1, default=float))


def rebuild(cfg, name, verbose=False):
    """Reconstruct a saved experiment: the Phase 1 weights, then the SAME PEFT
    structure the run used, then its trained tensors.

    The structure has to be rebuilt before loading, because a LoRA factor or an
    adapter has nowhere to land in a plain encoder. The technique and its settings
    were saved alongside the weights precisely so this can be done without the
    caller having to remember them.

    For QLoRA the NF4 base is re-quantized from the Phase 1 weights first. That is
    exact, not an approximation: the quantized base is frozen for the whole run, so
    re-quantizing the same source weights reproduces the same packed bytes, and the
    checkpoint then overwrites them with its own identical copy."""
    p = Path(cfg.out_dir) / "models" / f"{name}.pt"
    if not p.exists():
        raise FileNotFoundError(f"no checkpoint for {name!r} at {p}")
    ck = torch.load(p, map_location="cpu", weights_only=False)
    saved = ck.get("cfg", {})
    c = replace(cfg, pool=saved.get("pool", cfg.pool),
                sim=saved.get("sim", cfg.sim),
                proj_dim=saved.get("proj_dim", cfg.proj_dim))
    keep = ck.get("backbone_cfg", {}).get("n_layers")
    model, bc, _ = load_phase1(c, n_layers_keep=keep, verbose=verbose)

    tech, info = ck.get("technique", ""), ck.get("info", {}) or {}
    t = tech.lower()
    if "lorma" in t:
        # checked before "lora": the two technique strings are easy to confuse,
        # and rebuilding a LoRMA run as LoRA would load [r, d_out] factors into
        # [r, d_in] tensors and fail only if the dimensions happen to differ
        inject_lorma(model, r=info.get("r", 16), alpha=info.get("alpha", 32),
                     targets=tuple(info.get("targets", ATTN_TARGETS)))
    elif "qlora" in t:
        quantize_nf4(model, targets=ATTN_TARGETS, ffn_targets=FFN_TARGETS)
        inject_lora(model, r=info.get("r", 16), alpha=info.get("alpha", 32),
                    targets=tuple(info.get("targets", ATTN_TARGETS)))
    elif "lora" in t:
        inject_lora(model, r=info.get("r", 16), alpha=info.get("alpha", 32),
                    targets=tuple(info.get("targets", ATTN_TARGETS)))
    elif "adapter" in t:
        inject_adapters(model, bottleneck=info.get("bottleneck", 64),
                        variant=info.get("variant", "pfeiffer"))
    elif "ia3" in t:
        inject_ia3(model)

    sd, cur = ck["model"], model.state_dict()
    bad = [k for k in sd if k in cur and cur[k].shape != sd[k].shape]
    missing = [k for k in cur if k not in sd and k not in ("rope_cos", "rope_sin")]
    if bad or missing:
        raise RuntimeError(
            f"{name}: the rebuilt structure does not match the checkpoint.\n"
            f"  shape mismatch: {bad[:6]}\n  absent from the file: {missing[:6]}\n"
            f"  technique was {tech!r} with {info}")
    model.load_state_dict(sd, strict=False)
    model.to(DEVICE).eval()
    if verbose:
        print(f"[{name}] rebuilt ({tech}), dev micro-F1@7 "
              f"{ck.get('metrics',{}).get('micro_f1', float('nan')):.4f}")
    return model, bc, ck


def write_submission(cfg, model, name, pad_id, test_data=None, tok=None):
    """prediction.json + submission.zip for one model.

    Format, from the task statement: a map from query id to EXACTLY 7 unique
    candidate ids, drawn from that split's pool, ordered most relevant first.
    Every requirement is asserted here rather than discovered by the leaderboard."""
    if test_data is None:
        test_data = prepare_split(cfg, tok, pad_id, "test")
    k = cfg.top_k
    if len(test_data["cid"]) < k:
        raise RuntimeError(
            f"the test pool has only {len(test_data['cid'])} candidates but the "
            f"format demands {k} unique ids per query")
    Zc = embed_all(model, test_data["Xc"], test_data["nc"], pad_id, cfg,
                   desc="test pool")
    Zq = embed_all(model, test_data["Xq"], test_data["nq"], pad_id, cfg,
                   desc="test queries")
    topk = rank_all(Zq, Zc, k=k)
    pred = {}
    for i, qid in enumerate(test_data["qid"]):
        ids = [test_data["cid"][j] for j in topk[i][:k]]
        assert len(ids) == k, f"{qid}: {len(ids)} ids, need exactly {k}"
        assert len(set(ids)) == k, f"{qid}: duplicate ids"
        pred[qid] = ids
    pool = set(test_data["cid"])
    assert all(c in pool for v in pred.values() for c in v), \
        "a predicted id is not in test_candidates.json"

    d = Path(cfg.out_dir) / "submissions" / name
    d.mkdir(parents=True, exist_ok=True)
    pj = d / "prediction.json"
    pj.write_text(json.dumps(pred, indent=1))
    zp = d / "submission.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, "prediction.json")
    print(f"[{name}] submission: {len(pred)} queries x {k} ids -> {zp}")
    return str(zp), pred

In [ ]:
def run_pcr(name, technique, setup, ecfg=None, n_layers_keep=None, kd=None,
            notebook_stem="notebook"):
    """setup(model, cfg) -> info dict. Called after the Phase 1 weights load and
    before training, so it can inject PEFT and set what is trainable."""
    c = replace(cfg, **(ecfg or {}))
    if already_done(c, name):
        return None, load_results(c)[name]["metrics"]
    set_seed(c.seed)
    t_start = time.time()

    model, bc, ck = load_phase1(c, n_layers_keep=n_layers_keep)
    info = setup(model, c) or {}
    model.to(DEVICE)
    tr, tot = param_report(model, f"[{name}] ")
    if tr == 0:
        raise RuntimeError(
            f"{name}: nothing is trainable. The setup froze the model and "
            f"injected no adapters — training would be a no-op.")

    c = autofit_batch(model, c, PAD_ID)
    start, skip = resume_into(model, c, name, len(PAIRS))
    if not skip:
        best, hist = train(model, TRAIN, DEV, PAIRS, NEGS0, PAD_ID, c,
                           label=name, start_step=start, kd=kd)
    else:
        best, hist = evaluate(model, DEV, PAD_ID, c)[0], []
        best["steps_trained"] = start; best["minutes"] = 0.0

    metrics, topk = evaluate(model, DEV, PAD_ID, c, desc="final")
    metrics["steps_trained"] = best.get("steps_trained", 0)
    metrics["minutes"] = best.get("minutes", 0.0)
    costs = {"ms_per_doc": measure_latency(model, PAD_ID, c),
             "trainable_mb": state_bytes(trainable_state(model)) / 1e6,
             "full_mb": state_bytes(model.state_dict()) / 1e6,
             "wall_min": (time.time() - t_start) / 60}
    save_run(c, name, model, bc, technique, info, metrics, tr, tot)
    save_dev_analysis(c, name, DEV, topk, metrics)
    record(c, name, technique, info, metrics, tr, tot, costs,
           extra={"history": hist})
    save_notebook(Path(c.out_dir) / "experiments" / name, notebook_stem)
    print(f"[{name}] {costs['wall_min']:.1f} min total, "
          f"{costs['ms_per_doc']:.1f} ms/doc at inference")
    return model, metrics


# ──────────────── what each technique changes, in one place ────────────────
# Defined here rather than in the experiment cells so that every notebook —
# including the studies and the analysis — has all of them available.

def setup_full(model, c):
    for p in model.parameters():
        p.requires_grad = True
    return {"all_params": True}


def setup_partial(model, c):
    idx = set_partial_trainable(model, last_k=4, train_norm=True)
    unfreeze_readout(model)
    print(f"  training blocks {idx} of {len(model.backbone.layers)} + final norm")
    return {"last_k": 4, "blocks": idx}


def setup_lora(model, c):
    freeze_all(model)
    mods = inject_lora(model, r=16, alpha=32, targets=ATTN_TARGETS)
    unfreeze_readout(model)
    print(f"  LoRA on {len(mods)} projections, r=16 alpha=32")
    return {"r": 16, "alpha": 32, "targets": list(ATTN_TARGETS),
            "n_modules": len(mods)}


def setup_lorma(model, c):
    freeze_all(model)
    mods = inject_lorma(model, r=16, alpha=32, targets=ATTN_TARGETS)
    unfreeze_readout(model)
    print(f"  LoRMA+ on {len(mods)} projections, r=16 alpha=32")
    return {"r": 16, "alpha": 32, "targets": list(ATTN_TARGETS),
            "n_modules": len(mods), "variant": LORMA_VARIANT["name"]}


def setup_adapters(model, c):
    freeze_all(model)
    mods = inject_adapters(model, bottleneck=64, variant="pfeiffer")
    unfreeze_readout(model)
    for m in mods:
        for p in m.parameters():
            p.requires_grad = True
    print(f"  {len(mods)} adapters, bottleneck 64, Pfeiffer placement")
    return {"bottleneck": 64, "variant": "pfeiffer", "n_adapters": len(mods)}


def setup_qlora(model, c):
    freeze_all(model)
    _, qstats = quantize_nf4(model, targets=ATTN_TARGETS,
                             ffn_targets=FFN_TARGETS)
    mods = inject_lora(model, r=16, alpha=32, targets=ATTN_TARGETS)
    unfreeze_readout(model)
    print(f"  LoRA on {len(mods)} NF4 projections, r=16")
    return {"r": 16, "alpha": 32, "nf4": qstats, "n_modules": len(mods)}


def setup_ia3(model, c):
    freeze_all(model)
    mods = inject_ia3(model)
    unfreeze_readout(model)
    print(f"  IA3 on {len(mods)} projections (k, v, ffn.up)")
    return {"targets": ["k", "v", "ffn.up"], "n_modules": len(mods)}


def setup_student(model, c):
    for p in model.parameters():
        p.requires_grad = True
    return {"keep_layers": len(model.backbone.layers)}


def identity_check(inject, label, n=4):
    """Every PEFT module here is an identity at step 0 — B = 0 for LoRA, W_up = 0
    for the adapter, ℓ = 1 for IA3. Assert it on real document embeddings rather
    than taking it on trust: a technique that silently perturbs the model at init
    is not being compared on the same footing as the others."""
    set_seed(cfg.seed)
    a, _, _ = load_phase1(cfg, verbose=False); a.to(DEVICE).eval()
    x = torch.from_numpy(DEV["Xq"][:n].astype(np.int64)).to(DEVICE)
    with torch.no_grad():
        z0 = a(x, x == PAD_ID).float().cpu()
    freeze_all(a)
    mods = inject(a)
    a.to(DEVICE).eval()
    with torch.no_grad():
        z1 = a(x, x == PAD_ID).float().cpu()
    gap = (z0 - z1).abs().max().item()
    print(f"{label} identity check: max|Δembedding| = {gap:.3e}  (must be ~0)")
    assert gap < 1e-4, f"{label} changed the model at initialisation"
    return a, mods

## 1 — Data

In [ ]:
cfg = P7Config(out_dir=str(OUT), cache_dir=str(OUT / "cache"),
               results_json=str(OUT / "results.json"),
               ckpt_p1=str(CKPT_P1), tokenizer_json=str(TOK_JSON),
               hf_token=HF_TOKEN, quick=QUICK, window=WIN, max_len=MAXL,
               max_minutes=BUDGET_MIN, embed_batch=8,
               backbone=BACKBONE, dec_causal=(DEC_ATTN == "causal"),
               bilstm_hidden=BILSTM_HID,
               pool=(POOL or ("last" if DEC_ATTN == "causal" else "mean")))
set_seed(cfg.seed)
tok, PAD_ID, VOCAB = load_tokenizer(cfg)
TRAIN = prepare_split(cfg, tok, PAD_ID, "train")
DEV   = prepare_split(cfg, tok, PAD_ID, "dev")
PAIRS = build_pairs(TRAIN)
NEGS0 = None          # mined lazily, once, below
print(f"\ntrain {len(TRAIN['qid'])} queries / {len(TRAIN['cid'])} candidates / "
      f"{len(PAIRS)} pairs    dev {len(DEV['qid'])} / {len(DEV['cid'])}")
_g = [len(g) for g in DEV["gold"]]
_cap = sum(min(cfg.top_k, len(p)) for p in DEV["pos"])
_P, _R = _cap / (cfg.top_k * len(_g)), _cap / sum(_g)
CEIL = 2 * _P * _R / max(_P + _R, 1e-12)
print(f"ceiling micro-F1@{cfg.top_k} = {CEIL:.4f}")

tokenizer: vocab 40,000  pad_id 0
[cache hit ] train: 827 queries, 4320 candidates (p7_train_head_512_0.50.npz)
[cache hit ] dev: 118 queries, 1023 candidates (p7_dev_head_512_0.50.npz)

train 827 queries / 4320 candidates / 5308 pairs    dev 118 / 1023
ceiling micro-F1@7 = 0.6181


## 2 — Chunked retrieval

The tokenization is model-independent, so it is cached to Drive once and reused
by every experiment and every session. Only the embedding is redone per model,
and it runs under `no_grad` in small batches — which is why a 16-chunk index of
the whole pool fits comfortably on a T4.

In [ ]:
def chunk_windows(text, k, max_len, overlap=0.2):
    """k windows SPANNING the document. Packing them from the front would mean
    the end of a long judgment is never read."""
    ids = enc_ids(tok, " ".join(text.split()[:40000]))
    L, T = max_len, len(ids)
    if T <= L:
        return [ids + [PAD_ID] * (L - len(ids))]
    stride = max(1, int(round(L * (1 - overlap))))
    n = min(1 + math.ceil((T - L) / stride), k)
    st = [0] if n == 1 else [int(round(i * (T - L) / (n - 1))) for i in range(n)]
    return [ids[s:s + L] + [PAD_ID] * max(0, L - len(ids[s:s + L])) for s in st]


def chunk_tokens(split, kind, texts, k):
    key = f"p7v2_{split}_{kind}_{cfg.max_len}_{k}"
    f = Path(cfg.cache_dir) / f"{key}.npz"
    if f.exists():
        z = np.load(f)
        print(f"  [cache hit ] {split}/{kind}: {len(z['X'])} chunks")
        return z["X"], z["owner"]
    X, owner, t0 = [], [], time.time()
    for i, t in enumerate(texts):
        for c in chunk_windows(t, k, cfg.max_len):
            X.append(c); owner.append(i)
        if i and i % 500 == 0:
            print(f"  chunking {split}/{kind} {i}/{len(texts)} "
                  f"({time.time()-t0:.0f}s)")
    X = np.asarray(X, dtype=np.int32); owner = np.asarray(owner, dtype=np.int64)
    _t = f.with_suffix(f".tmp{os.getpid()}.npz")      # atomic for parallel runs
    np.savez_compressed(_t, X=X, owner=owner); _t.replace(f)
    print(f"  [tokenized ] {split}/{kind}: {len(texts)} docs -> {len(X)} chunks "
          f"({time.time()-t0:.0f}s, cached)")
    return X, owner


def chunk_rank(model, Xq, qo, Xc, co, nq, nc, k=20, agg=AGG, desc=""):
    Zc = embed_all(model, Xc, None, PAD_ID, cfg, desc=desc and f"{desc} pool")
    Zq = embed_all(model, Xq, None, PAD_ID, cfg)
    out, cstart = [], np.searchsorted(co, np.arange(nc))
    cnt = np.bincount(co, minlength=nc)
    for lo in range(0, nq, 64):                 # blocked: the full score matrix
        hi = min(lo + 64, nq)                   # is never materialised
        sel = np.where((qo >= lo) & (qo < hi))[0]
        S = (Zq[sel] @ Zc.T).astype(np.float32)
        M = (np.maximum.reduceat(S, cstart, axis=1) if agg == "max"
             else np.add.reduceat(S, cstart, axis=1)
                  / np.maximum(cnt, 1)[None, :])
        rows = qo[sel] - lo
        qs = np.searchsorted(rows, np.arange(hi - lo))
        Q = (np.maximum.reduceat(M, qs, axis=0) if agg == "max"
             else np.add.reduceat(M, qs, axis=0)
                  / np.maximum(np.bincount(rows, minlength=hi - lo), 1)[:, None])
        out.append(np.argsort(-Q, axis=1)[:, :k])
    return np.concatenate(out)


DEVQ = load_queries(cfg, "dev"); DEVC = load_candidates(cfg, "dev")
_qm = {q: t for q, t, _ in DEVQ}; _cm = {c: t for c, t in DEVC}
DQX, DQO = chunk_tokens("dev", "q", [_qm[q] for q in DEV["qid"]], KQ)
DCX, DCO = chunk_tokens("dev", "c", [_cm[c] for c in DEV["cid"]], KC)


def eval_chunked(model, desc=""):
    tk = chunk_rank(model, DQX, DQO, DCX, DCO,
                    len(DEV["qid"]), len(DEV["cid"]), k=20, desc=desc)
    m = micro_f1_at_k(tk, DEV["gold"], DEV["cid"], cfg.top_k, DEV["pos"])
    m.update(recall_at(tk, DEV["gold"], DEV["cid"]))
    return m, tk

[cache hit ] dev queries: 118 (p7_raw_dev_queries.json.gz)
[cache hit ] dev candidates: 1023 (p7_raw_dev_candidates.json.gz)
  [cache hit ] dev/q: 1387 chunks
  [cache hit ] dev/c: 10970 chunks


## 3 — The experiments

Phase 7 requires at least 4 substantially different experiments, at least 3
distinct techniques from its list, **LoRMA**, and at least one knowledge
distillation. This set clears all four: full fine-tuning, LoRMA, IA³ and KD,
which is 4 techniques from the list plus the distillation control.

Each one trains, is scored both ways, and writes its submission before the next
begins — so an interrupted session still leaves you with uploadable files.

In [ ]:
def initial_negatives(cfg, pad_id):
    """Mined once from the untrained encoder and cached to Drive — every
    technique then starts from the same training signal, so a score difference
    is the technique and not a luckier draw of negatives."""
    key = (f"{cfg.window}_{cfg.max_len}_{cfg.hard_pool}"
           + (f"_q{cfg.max_train}" if cfg.max_train else "")
           + (f"_c{cfg.max_cand}" if cfg.max_cand else ""))
    f = Path(cfg.cache_dir) / f"p7v2_negs_{key}.json"
    if f.exists():
        print(f"  [cache hit ] initial hard negatives")
        return json.loads(f.read_text())
    print("  mining initial hard negatives from the untrained encoder (once)")
    m, _, _ = load_phase1(cfg, verbose=False)
    m.to(DEVICE)
    negs, _ = mine_hard_negatives(m, TRAIN, pad_id, cfg)
    del m
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    _t = f.with_suffix(f".tmp{os.getpid()}.json")
    _t.write_text(json.dumps(negs)); _t.replace(f)
    return negs


SETUPS = {
    "full_ft": ("full fine-tuning", setup_full, {"lr": 5e-5}),
    "lorma":   ("LoRMA+ (from scratch, multiplicative)", setup_lorma, {"lr": 3e-4}),
    "lora":    ("LoRA (from scratch)", setup_lora, {"lr": 3e-4}),
    "ia3":     ("IA3 (from scratch)", setup_ia3, {"lr": 5e-3}),
    "adapters":("bottleneck adapters (from scratch)", setup_adapters, {"lr": 3e-4}),
    "qlora":   ("QLoRA (NF4 from scratch + LoRA)", setup_qlora, {"lr": 3e-4}),
}
TEST = None


def submit(model, name):
    """Chunked submission — the configuration A measured as best."""
    global TEST
    if TEST is None:
        TEST = prepare_split(cfg, tok, PAD_ID, "test")
    tq = {q: t for q, t, _ in load_queries(cfg, "test")}
    tc = {c: t for c, t in load_candidates(cfg, "test")}
    TQX, TQO = chunk_tokens("test", "q", [tq[q] for q in TEST["qid"]], KQ)
    TCX, TCO = chunk_tokens("test", "c", [tc[c] for c in TEST["cid"]], KC)
    tk = chunk_rank(model, TQX, TQO, TCX, TCO,
                    len(TEST["qid"]), len(TEST["cid"]), k=cfg.top_k, desc="test")
    pred = {}
    for i, qid in enumerate(TEST["qid"]):
        ids = [TEST["cid"][j] for j in tk[i][:cfg.top_k]]
        assert len(set(ids)) == cfg.top_k, f"{qid}: need 7 unique ids"
        pred[qid] = ids
    assert set(pred) == set(TEST["qid"]), "a test query has no prediction"
    d = Path(cfg.out_dir) / "submissions" / name
    d.mkdir(parents=True, exist_ok=True)
    (d / "prediction.json").write_text(json.dumps(pred, indent=1))
    with zipfile.ZipFile(d / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(d / "prediction.json", "prediction.json")
    import hashlib
    h = hashlib.sha1(json.dumps(pred, sort_keys=True).encode()).hexdigest()[:10]
    print(f"  submission -> {d/'submission.zip'}   (hash {h})")
    return h


def run_one(key):
    global NEGS0
    tech, setup, over = SETUPS[key]
    name = f"E_{key}"
    if already_done(cfg, name):
        return
    if NEGS0 is None:
        NEGS0 = initial_negatives(cfg, PAD_ID)
    print(f"\n{'='*68}\n{name}  —  {tech}\n{'='*68}")
    set_seed(cfg.seed); t0 = time.time()
    c = replace(cfg, **over)
    model, bc, _ = load_phase1(c)
    info = setup(model, c) or {}
    model.to(DEVICE)
    c = autofit_batch(model, c, PAD_ID)
    tr, tot = param_report(model, f"[{name}] ")
    start, skip = resume_into(model, c, name, len(PAIRS))
    if not skip:
        # selection runs on the SINGLE-window metric: a chunked evaluation costs
        # ~130 s on a T4 and would eat the budget. The representation is already
        # decided, so this only picks a checkpoint.
        best, hist = train(model, TRAIN, DEV, PAIRS, NEGS0, PAD_ID, c,
                           label=name, start_step=start)
    else:
        best, hist = evaluate(model, DEV, PAD_ID, c)[0], []
        best["steps_trained"] = start; best["minutes"] = 0.0
    single, _ = evaluate(model, DEV, PAD_ID, c)
    chunked, topk = eval_chunked(model, desc=name)
    chunked["single_window"] = single["micro_f1"]
    chunked["steps_trained"] = best.get("steps_trained", 0)
    chunked["minutes"] = best.get("minutes", 0.0)
    print(f"  [{name}] single window {single['micro_f1']:.4f}  ->  "
          f"chunked {KQ}x{KC} {AGG} {chunked['micro_f1']:.4f}")
    costs = {"ms_per_doc": measure_latency(model, PAD_ID, c),
             "trainable_mb": state_bytes(trainable_state(model)) / 1e6,
             "wall_min": (time.time() - t0) / 60}
    save_run(c, name, model, bc, tech, info, chunked, tr, tot)
    save_dev_analysis(c, name, DEV, topk, chunked)
    record(c, name, tech, info, chunked, tr, tot, costs,
           extra={"history": hist, "retrieval": {"kq": KQ, "kc": KC, "agg": AGG}})
    submit(model, name)
    save_notebook(Path(cfg.out_dir) / "experiments" / name, "notebook")
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

## 4 — A cross-encoder reranker

Everything else in this project is a **bi-encoder**: query and candidate are
embedded independently and compared by cosine. That is what makes a 1,023-document
search cheap, and it is also what caps it — the model never sees the two
documents together, so it can never notice that *this* paragraph of the query is
about the same statute as *that* paragraph of the candidate.

A cross-encoder reads the pair jointly. Attention runs across both halves, so
correspondence is something the model can represent directly rather than
approximate with a dot product. The cost is that scoring is O(queries ×
candidates) forward passes, which is why it runs only as a **second stage**, on
the top-50 a cheap lexical system has already produced.

Three reasons this is the most promising thing left:

**Far more supervision.** The bi-encoder trains on 5,308 (query, positive) pairs.
Reranking the lexical top-50 gives 827 x 50 ≈ 41,000 labelled pairs, and every
negative is a genuinely confusable case rather than a random draw.

**An easier problem.** Reordering 50 candidates is not the same task as being a
global metric over 1,023. Your bi-encoder plateaus at 0.28 — the same as the
untrained checkpoint — because 827 queries cannot teach a global similarity
metric. Local reordering is within reach of that much data.

**It attacks the actual gap.** Measured on train, a tuned bag of words scores
0.3311 and the dense encoder 0.2574-0.2924. A reranker does not have to beat
lexical on its own; it only has to improve the ordering lexical hands it.

The honest risk, stated up front: the Phase 1 model is small and trained on
1.1B tokens. A cross-encoder built on a weak LM may still not beat the lexical
ordering it is reranking. If it does not, that is a publishable negative result
for §15.2 — and the notebook reports it rather than hiding it, because it always
compares against the first stage it started from.

In [ ]:
# ─────────── stage 1: lexical retrieval, the pool the reranker reorders ───────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

RERANK_DEPTH = 50      # candidates per query handed to the cross-encoder
NGRAM        = (2, 3)  # measured best on train: 0.3311 vs 0.2574 for unigrams
# A PAIR occupies one sequence, so it cannot be longer than the RoPE table
# Phase 1 precomputed — which is cfg.max_len, not 512 by assumption.
CE_LEN       = cfg.max_len
N_NEG        = 7       # negatives per positive in a training group
CE_EPOCHS    = 3
CE_LR        = 2e-5
CE_BATCH     = 8       # GROUPS per step = CE_BATCH * (1 + N_NEG) sequences


def lex_scores(qtexts, ctexts, ngram=NGRAM, min_df=3, desc=""):
    tv = TfidfVectorizer(lowercase=True, sublinear_tf=True, stop_words="english",
                         min_df=min_df, ngram_range=ngram, dtype=np.float32)
    X = normalize(tv.fit_transform(list(ctexts) + list(qtexts)))
    S = np.asarray((X[len(ctexts):] @ X[:len(ctexts)].T).todense(), np.float32)
    print(f"  {desc}lexical {ngram[0]}-{ngram[1]}gram {S.shape}")
    return S


TRQ = load_queries(cfg, "train"); TRC = load_candidates(cfg, "train")
_tq = {q: t for q, t, _ in TRQ}; _tc = {c: t for c, t in TRC}
TRQT = [_tq[q] for q in TRAIN["qid"]]; TRCT = [_tc[c] for c in TRAIN["cid"]]
DQT = [_qm[q] for q in DEV["qid"]];    DCT = [_cm[c] for c in DEV["cid"]]

LEX_TR = lex_scores(TRQT, TRCT, desc="train ")
LEX_DV = lex_scores(DQT, DCT, desc="dev ")
_tk = np.argsort(-LEX_DV, axis=1)[:, :20]
LEX_DEV_F1 = micro_f1_at_k(_tk, DEV["gold"], DEV["cid"], cfg.top_k)["micro_f1"]
print(f"\nSTAGE 1 dev micro-F1@{cfg.top_k} = {LEX_DEV_F1:.4f}  "
      f"<- the number the reranker must beat")


def pool_of(S, gold_idx=None, depth=RERANK_DEPTH):
    """Top-`depth` candidate indices per query. During TRAINING the known
    positives are forced in even when lexical missed them — otherwise the model
    only ever sees the positives lexical already ranks highly and learns to
    agree with it instead of correcting it."""
    out = []
    for i in range(S.shape[0]):
        top = np.argpartition(-S[i], depth)[:depth]
        if gold_idx is not None and len(gold_idx[i]):
            top = np.union1d(top, np.fromiter(gold_idx[i], int, len(gold_idx[i])))
        out.append(top.astype(np.int32))
    return out


TR_POS = [set(int(x) for x in p) for p in TRAIN["pos"]]
POOL_TR = pool_of(LEX_TR, TR_POS)
POOL_DV = pool_of(LEX_DV)
_rec = np.mean([len(set(POOL_DV[i]) & {DEV["cid"].index(c) for c in DEV["gold"][i]
                                      if c in DEV["cid"]}) /
                max(1, len([c for c in DEV["gold"][i] if c in DEV["cid"]]))
                for i in range(len(DEV["qid"]))])
print(f"stage-1 recall@{RERANK_DEPTH} on dev = {_rec:.4f}   "
      f"<- the reranker's ceiling: it can only reorder what lexical retrieved")

[cache hit ] train queries: 827 (p7_raw_train_queries.json.gz)
[cache hit ] train candidates: 4320 (p7_raw_train_candidates.json.gz)
  train lexical 2-3gram (827, 4320)
  dev lexical 2-3gram (118, 1023)

STAGE 1 dev micro-F1@7 = 0.4202  <- the number the reranker must beat
stage-1 recall@50 on dev = 0.8182   <- the reranker's ceiling: it can only reorder what lexical retrieved


### 4.1 Which 512 tokens?

A pair must fit in 512 tokens, so roughly 256 each. Picking the two heads would
be the obvious choice and the wrong one — the evidence notebook measured a
single head window at 0.1968 against 0.2924 for chunked retrieval, so heads are
the weakest view of these documents.

Instead each document is cut into chunks and the pair is chosen **lexically**:
the (query chunk, candidate chunk) with the highest TF-IDF overlap. The
cross-encoder then spends its 512 tokens on the passages most likely to contain
the correspondence, and the choice costs no GPU time.

In [ ]:
CE_CHUNKS = 8          # chunks per document considered when choosing the pair
# half the pair budget, converted to words: this corpus runs ~1.3 BPE tokens
# per word, so asking for more would just be truncated away inside ce_ids()
CE_WORDS = max(20, int((CE_LEN // 2) / 1.3))
print(f"pair budget {CE_LEN} tokens -> {CE_WORDS} words per side, "
      f"{CE_CHUNKS} chunks considered per document")


def word_chunks(text, k=CE_CHUNKS, w=CE_WORDS):
    ws = text.split()
    if len(ws) <= w:
        return [" ".join(ws)] or [""]
    starts = np.linspace(0, max(0, len(ws) - w), k).astype(int)
    seen, out = set(), []
    for s in starts:
        if s in seen:
            continue
        seen.add(s); out.append(" ".join(ws[s:s + w]))
    return out


def best_pairs(qtexts, ctexts, pools, desc=""):
    """For each (query, candidate) in the pool, the chunk pair with the highest
    TF-IDF cosine. Returns {(qi, ci): (q_chunk_text, c_chunk_text)}."""
    t0 = time.time()
    qch = [word_chunks(t) for t in qtexts]
    cch = [word_chunks(t) for t in ctexts]
    flat = [c for ch in cch for c in ch] + [c for ch in qch for c in ch]
    tv = TfidfVectorizer(lowercase=True, sublinear_tf=True, stop_words="english",
                         min_df=2, ngram_range=(1, 2), dtype=np.float32)
    X = normalize(tv.fit_transform(flat))
    nc = sum(len(ch) for ch in cch)
    Xc, Xq = X[:nc], X[nc:]
    coff = np.cumsum([0] + [len(ch) for ch in cch])
    qoff = np.cumsum([0] + [len(ch) for ch in qch])
    out = {}
    for qi in range(len(qtexts)):
        A = Xq[qoff[qi]:qoff[qi + 1]]
        for ci in pools[qi]:
            ci = int(ci)
            B = Xc[coff[ci]:coff[ci + 1]]
            M = np.asarray((A @ B.T).todense())
            a, b = np.unravel_index(int(M.argmax()), M.shape)
            out[(qi, ci)] = (qch[qi][a], cch[ci][b])
        if qi and qi % 200 == 0:
            print(f"    {desc}{qi}/{len(qtexts)} ({time.time()-t0:.0f}s)")
    print(f"  {desc}chose {len(out):,} chunk pairs in {time.time()-t0:.0f}s")
    return out


PAIR_TR = best_pairs(TRQT, TRCT, POOL_TR, "train ")
PAIR_DV = best_pairs(DQT, DCT, POOL_DV, "dev ")

pair budget 512 tokens -> 196 words per side, 8 chunks considered per document
    train 200/827 (22s)
    train 400/827 (31s)
    train 600/827 (41s)
    train 800/827 (50s)
  train chose 42,873 chunk pairs in 51s
  dev chose 5,900 chunk pairs in 5s


### 4.2 The model

The Phase 1 backbone with a scalar head. No `[SEP]` token was pretrained, so the
two halves are joined by whatever special token the tokenizer does have — the
model has no segment embeddings either way, and with RoPE the boundary is
learnable from position alone.

In [ ]:
_sp = tok.encode("", add_special_tokens=True)
SEP_ID = int(_sp[0]) if len(_sp) else PAD_ID
print(f"joining the two halves with token id {SEP_ID} "
      f"({'tokenizer special token' if len(_sp) else 'pad id — no special tokens found'})")


def ce_ids(qtext, ctext, total=CE_LEN):
    half = (total - 1) // 2
    a = enc_ids(tok, qtext)[:half]
    b = enc_ids(tok, ctext)[:total - 1 - len(a)]
    ids = a + [SEP_ID] + b
    return ids + [PAD_ID] * (total - len(ids))


class CrossEncoder(nn.Module):
    """Phase 1 backbone -> masked mean -> scalar relevance. One number per pair."""
    def __init__(self, backbone, d_model, dropout=0.1):
        super().__init__()
        self.backbone = backbone
        self.drop = nn.Dropout(dropout)
        self.score = nn.Linear(d_model, 1)
        nn.init.normal_(self.score.weight, std=0.02)
        nn.init.zeros_(self.score.bias)

    def forward(self, ids, pad_mask):
        h = self.backbone(ids, pad_mask)
        keep = (~pad_mask).unsqueeze(-1).to(h.dtype)
        z = (h * keep).sum(1) / keep.sum(1).clamp(min=1)
        return self.score(self.drop(z)).squeeze(-1)


def build_ce(c):
    m, bc, _ = load_phase1(c, verbose=True)
    ce = CrossEncoder(m.backbone, bc.d_model, c.dropout)
    ce.backbone.grad_checkpoint = True
    return ce.to(DEVICE), bc

joining the two halves with token id 2 (tokenizer special token)


### 4.3 Training

Each step is a **group**: one positive and `N_NEG` negatives drawn from that
query's lexical pool, scored together, with a softmax cross-entropy over the
group. That is a listwise objective — it asks the model to rank the positive
above these specific hard negatives, which is exactly the job at inference.

In [ ]:
def ce_groups(pools, pos_sets, n_neg=N_NEG, seed=0):
    rng = np.random.default_rng(seed)
    g = []
    for qi, pool in enumerate(pools):
        pos = [c for c in pool if int(c) in pos_sets[qi]]
        neg = [c for c in pool if int(c) not in pos_sets[qi]]
        if not pos or len(neg) < n_neg:
            continue
        for p in pos:
            g.append((qi, int(p),
                      [int(x) for x in rng.choice(neg, n_neg, replace=False)]))
    rng.shuffle(g)
    return g


def ce_batch(groups, pairs):
    ids = []
    for qi, p, negs in groups:
        for c in [p] + negs:
            ids.append(ce_ids(*pairs[(qi, c)]))
    return torch.tensor(ids, dtype=torch.long)


@torch.no_grad()
def ce_rank(ce, pools, pairs, nq, bs=32, desc=""):
    """Scores every pooled pair and returns, per query, its candidates ordered
    by the cross-encoder."""
    ce.eval()
    order, t0 = [], time.time()
    for qi in range(nq):
        pool = [int(c) for c in pools[qi]]
        sc = []
        for i in range(0, len(pool), bs):
            ids = torch.tensor([ce_ids(*pairs[(qi, c)]) for c in pool[i:i + bs]],
                               dtype=torch.long, device=DEVICE)
            with torch.amp.autocast(device_type=DEVICE.type,
                                    enabled=(DEVICE.type == "cuda")):
                sc.append(ce(ids, ids == PAD_ID).float().cpu())
        s = torch.cat(sc).numpy()
        order.append([pool[j] for j in np.argsort(-s)])
        if qi and qi % 50 == 0:
            print(f"    {desc}{qi}/{nq} ({time.time()-t0:.0f}s)")
    ce.train()
    return order


def ce_eval(ce, desc=""):
    od = ce_rank(ce, POOL_DV, PAIR_DV, len(DEV["qid"]), desc=desc)
    tk = np.array([o[:20] + [o[-1]] * max(0, 20 - len(o)) for o in od])
    return micro_f1_at_k(tk, DEV["gold"], DEV["cid"], cfg.top_k), tk


def train_ce(ce, epochs=CE_EPOCHS, lr=CE_LR, batch=CE_BATCH, label="rerank"):
    G = ce_groups(POOL_TR, TR_POS)
    spe = max(1, len(G) // batch)
    total = spe * epochs
    opt = torch.optim.AdamW(
        [p for p in ce.parameters() if p.requires_grad], lr=lr,
        betas=(0.9, 0.98), eps=1e-6, weight_decay=0.01)
    warm = max(1, int(0.1 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm
        else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))
    print(f"  {label}: {len(G):,} groups, {batch} groups/step x "
          f"{1+N_NEG} docs = {batch*(1+N_NEG)} sequences, {spe} steps/epoch, "
          f"{epochs} epochs")
    best, best_sd, hist, t0 = {"micro_f1": -1.0}, None, [], time.time()
    for ep in range(1, epochs + 1):
        ce.train()
        rng = np.random.default_rng(ep); rng.shuffle(G)
        losses = []
        for si in range(spe):
            gb = G[si * batch:(si + 1) * batch]
            if not gb:
                continue
            ids = ce_batch(gb, PAIR_TR).to(DEVICE)
            with torch.amp.autocast(device_type=DEVICE.type,
                                    enabled=(DEVICE.type == "cuda")):
                s = ce(ids, ids == PAD_ID).view(len(gb), 1 + N_NEG)
                loss = F.cross_entropy(
                    s, torch.zeros(len(gb), dtype=torch.long, device=DEVICE))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(
                [p for p in ce.parameters() if p.requires_grad], 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            losses.append(float(loss.detach()))
            if si and si % 200 == 0:
                print(f"    ep{ep} step {si}/{spe} loss "
                      f"{np.mean(losses[-200:]):.4f} "
                      f"({(time.time()-t0)/60:.1f} min)")
        m, _ = ce_eval(ce, f"ep{ep} ")
        hist.append({"epoch": ep, "loss": float(np.mean(losses)),
                     "micro_f1": m["micro_f1"]})
        mark = ""
        if m["micro_f1"] > best["micro_f1"]:
            best = {**m, "epoch": ep}
            best_sd = {k: v.detach().cpu().clone() for k, v in ce.state_dict().items()}
            mark = "  <- best"
        print(f"  epoch {ep}/{epochs}  loss {np.mean(losses):.4f}  "
              f"dev micro-F1@{cfg.top_k} {m['micro_f1']:.4f}  "
              f"(stage 1 was {LEX_DEV_F1:.4f})  "
              f"({(time.time()-t0)/60:.1f} min){mark}")
    if best_sd:
        ce.load_state_dict(best_sd)
    return best, hist

### 4.4 Run, compare honestly, submit

The reranker is only used for the submission **if it beat its own first stage on
dev**. If it did not, the lexical ordering is submitted instead and the notebook
says so — a reranker that cannot improve on a bag of words is a result, not a
thing to paper over.

In [ ]:
def run_reranker(name="E_rerank", epochs=CE_EPOCHS, lr=CE_LR):
    if already_done(cfg, name):
        print(f"{name} already recorded — skipping"); return
    print(f"\n{'='*68}\n{name}  —  cross-encoder reranker over lexical top-"
          f"{RERANK_DEPTH}\n{'='*68}")
    set_seed(cfg.seed); t0 = time.time()
    ce, bc = build_ce(cfg)
    tr, tot = param_report(ce, f"[{name}] ")
    best, hist = train_ce(ce, epochs=epochs, lr=lr, label=name)
    won = best["micro_f1"] > LEX_DEV_F1
    print(f"\n  reranker {best['micro_f1']:.4f}  vs  stage-1 lexical "
          f"{LEX_DEV_F1:.4f}  ->  "
          + ("RERANKER WINS, it will be used for the submission"
             if won else
             "reranker did NOT beat the bag of words. Submitting the lexical "
             "ordering instead; report this, it is a real finding."))
    metrics = {**best, "stage1_lexical": float(LEX_DEV_F1),
               "beat_stage1": bool(won), "steps_trained": 0,
               "minutes": (time.time() - t0) / 60}
    record(cfg, name, f"cross-encoder reranker, lexical top-{RERANK_DEPTH}",
           {"depth": RERANK_DEPTH, "ngram": list(NGRAM), "n_neg": N_NEG,
            "ce_len": CE_LEN, "epochs": epochs, "lr": lr},
           metrics, tr, tot, {"wall_min": (time.time() - t0) / 60},
           extra={"history": hist})
    torch.save({"model": ce.state_dict()},
               Path(cfg.out_dir) / "models" / f"{name}.pt")
    submit_rerank(ce, name, use_ce=won)
    del ce
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


def submit_rerank(ce, name, use_ce=True):
    TEST = prepare_split(cfg, tok, PAD_ID, "test")
    tq = {q: t for q, t, _ in load_queries(cfg, "test")}
    tc = {c: t for c, t in load_candidates(cfg, "test")}
    TQT = [tq[q] for q in TEST["qid"]]; TCT = [tc[c] for c in TEST["cid"]]
    L = lex_scores(TQT, TCT, desc="test ")
    pools = pool_of(L, None, min(RERANK_DEPTH, len(TEST["cid"]) - 1))
    if use_ce:
        pairs = best_pairs(TQT, TCT, pools, "test ")
        order = ce_rank(ce, pools, pairs, len(TEST["qid"]), desc="test ")
    else:
        order = [list(np.argsort(-L[i])) for i in range(len(TEST["qid"]))]
    pred = {}
    for i, q in enumerate(TEST["qid"]):
        ids = [TEST["cid"][j] for j in order[i][:cfg.top_k]]
        assert len(set(ids)) == cfg.top_k, f"{q}: need {cfg.top_k} unique ids"
        pred[q] = ids
    assert set(pred) == set(TEST["qid"]), "a test query has no prediction"
    d = Path(cfg.out_dir) / "submissions" / name
    d.mkdir(parents=True, exist_ok=True)
    (d / "prediction.json").write_text(json.dumps(pred, indent=1))
    with zipfile.ZipFile(d / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(d / "prediction.json", "prediction.json")
    import hashlib
    h = hashlib.sha1(json.dumps(pred, sort_keys=True).encode()).hexdigest()[:10]
    print(f"  submission ({'reranked' if use_ce else 'lexical only'}) "
          f"-> {d/'submission.zip'}  (hash {h})")

In [ ]:
# ─────────── run ───────────
run_reranker(name="E_rerank", epochs=CE_EPOCHS, lr=CE_LR)


E_rerank  —  cross-encoder reranker over lexical top-50
MHA ENCODER (bidirectional, MLM-pretrained): 768d x 14L, d_ff=2048, vocab=40,001
  loaded 128/128 tensors (step 29000, val_loss 1.69698847756069)
  tok_emb std = 0.0267 (fresh init ~0.0200)
  bi-encoder: pool=mean sim=cos dim=768
[E_rerank] trainable 129,834,241 / 129,834,241 = 100.000%
  E_rerank: 5,308 groups, 8 groups/step x 8 docs = 64 sequences, 663 steps/epoch, 3 epochs
    ep1 step 200/663 loss 2.0559 (1.7 min)
    ep1 step 400/663 loss 2.0319 (3.4 min)
    ep1 step 600/663 loss 1.9788 (5.1 min)
    ep1 50/118 (6s)
    ep1 100/118 (11s)
  epoch 1/3  loss 2.0175  dev micro-F1@7 0.1823  (stage 1 was 0.4202)  (5.8 min)  <- best
    ep2 step 200/663 loss 1.8010 (7.5 min)
    ep2 step 400/663 loss 1.7507 (9.2 min)
    ep2 step 600/663 loss 1.6653 (10.9 min)
    ep2 50/118 (6s)
    ep2 100/118 (11s)
  epoch 2/3  loss 1.7323  dev micro-F1@7 0.2135  (stage 1 was 0.4202)  (11.6 min)  <- best
    ep3 step 200/663 loss 1.3977 (13.3 m